# Классификация болезней растений

## 1. Установка библиотек

In [ ]:
import os
import sys
import subprocess
import time

SESSION_STARTED = globals().get("SESSION_STARTED", time.time())
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

packages = [
    "timm==1.0.20",
    "ImageHash==4.3.2",
    "onnx==1.18.0",
    "onnxruntime-gpu==1.22.0",
    "scikit-learn>=1.4,<1.8",
    "pandas>=2.1,<3",
    "PyYAML>=6,<7",
    "cloudpickle>=2.2,<4",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", *packages],
    check=True,
    timeout=900,
)


## 2. Импорты

In [ ]:
from __future__ import annotations

import argparse
import ast
import contextlib
import copy
import hashlib
import inspect
import io
import json
import math
import multiprocessing
import os
import random
import re
import shutil
import signal
import socket
import subprocess
import sys
import time
import traceback
import urllib.request
import zipfile
from collections import Counter
from contextlib import nullcontext
from dataclasses import dataclass
from datetime import timedelta
from pathlib import Path, PurePosixPath
from typing import Sequence

import cloudpickle
import nbformat
import numpy as np
import pandas as pd
import torch
import torch.distributed as dist
import yaml
from IPython.display import FileLink, display
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
from torch import nn
from torch.nn import functional as F
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, Dataset, Sampler
from torchvision import transforms as T


## 3. Параметры эксперимента

In [ ]:
from pathlib import Path
import yaml

CONFIG_YAML = """\
seed: 42
input_root: /kaggle/input
work_dir: /kaggle/working/hotbed_field_v3_standalone
extra_manifest: auto
coco_stuff_negatives: true
auto_download_coco_masks: true
mask_download_minutes: 12
coco_negative_counts:
  train: 4000
  calibration: 800
  field_test: 800
negative_prepare_minutes: 10
backbone: vit_base_patch16_dinov3.lvd1689m
embedding_dim: 384
dinov3_pretrained_file: null
workers: 2
resume: true
minimum_negative_per_split: 50
minimum_coverage: 0.85
maximum_false_healthy: 0.05
calibration_target_accuracy: 0.95
calibration_max_negative_acceptance: 0.02
calibration_min_plant_recall: 0.98
prototype_centers: 3
prototype_samples_per_class_source: 100
train_cap_per_class_source: 2000
plantvillage_train_cap: 500
near_duplicate_bits: 4
session_budget_hours: 11.0
finalization_reserve_hours: 2.0
max_samples_per_epoch: 8192
candidates:
  - name: dinov3_384
    imgsz: 384
    seed: 42
    epochs: 24
    patience: 5
    batch_size: 4
    accum_steps: 4
    unfreeze_blocks: 6
    backbone_lr: 0.000006
    head_lr: 0.00010
    budget_fraction: 0.75
    init: pretrained
  - name: dinov3_512_refine
    imgsz: 512
    seed: 42
    epochs: 10
    patience: 4
    batch_size: 2
    accum_steps: 8
    unfreeze_blocks: 6
    backbone_lr: 0.000003
    head_lr: 0.00005
    budget_fraction: 1.0
    init: best_previous
"""
config = yaml.safe_load(CONFIG_YAML)
WORK = Path(config["work_dir"])
WORK.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = WORK / "field_v3.yaml"
CONFIG_PATH.write_text(
    yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding="utf-8"
)
print(CONFIG_PATH)


## 4. Загрузка и подготовка данных

In [ ]:
CLASSES = [
    "tomato_healthy",
    "tomato_bacterial_spot",
    "tomato_early_blight",
    "tomato_late_blight",
    "tomato_leaf_mold",
    "tomato_septoria",
    "tomato_yellow_leaf_curl_virus",
    "tomato_mosaic_virus",
    "tomato_powdery_mildew",
    "tomato_leaf_miner",
    "tomato_whitefly",
    "cucumber_healthy",
    "cucumber_downy_mildew",
    "cucumber_powdery_mildew",
    "cucumber_anthracnose",
    "cucumber_bacterial_wilt",
    "cucumber_belly_rot",
    "cucumber_pythium_fruit_rot",
    "cucumber_gummy_stem_blight",
    "pepper_healthy",
    "pepper_bacterial_spot",
    "pepper_cercospora_leaf_spot",
    "pepper_leaf_curl",
    "pepper_nutrient_deficiency",
    "pepper_powdery_mildew",
    "eggplant_healthy",
    "eggplant_insect_pest",
    "eggplant_leaf_spot",
    "eggplant_mosaic_virus",
    "eggplant_small_leaf",
    "eggplant_white_mold",
    "eggplant_wilt",
    "strawberry_healthy",
    "strawberry_leaf_scorch",
    "strawberry_calcium_deficiency",
    "strawberry_gray_mold",
    "strawberry_leaf_spot",
    "strawberry_powdery_mildew_leaf",
    "strawberry_anthracnose_fruit_rot",
]

ALIASES = {
    "tomato_healthy": [
        "tomato healthy",
        "tomato___healthy",
        "healthy tomato",
        "tomato leaf",
    ],
    "tomato_bacterial_spot": [
        "tomato bacterial spot",
        "tomato___bacterial_spot",
        "bacterial spot tomato",
        "tomato leaf bacterial spot",
    ],
    "tomato_early_blight": [
        "tomato early blight",
        "tomato___early_blight",
        "tomato early blight leaf",
    ],
    "tomato_late_blight": [
        "tomato late blight",
        "tomato___late_blight",
        "tomato leaf late blight",
    ],
    "tomato_leaf_mold": ["tomato leaf mold", "tomato___leaf_mold", "tomato mold leaf"],
    "tomato_septoria": [
        "tomato septoria leaf spot",
        "tomato___septoria_leaf_spot",
        "tomato septoria",
    ],
    "tomato_yellow_leaf_curl_virus": [
        "tomato yellow leaf curl virus",
        "tomato___tomato_yellow_leaf_curl_virus",
        "yellow leaf curl tomato",
        "tomato leaf yellow virus",
    ],
    "tomato_mosaic_virus": [
        "tomato mosaic virus",
        "tomato___tomato_mosaic_virus",
        "tomato leaf mosaic virus",
    ],
    "tomato_powdery_mildew": ["tomato powdery mildew", "powdery mildew tomato"],
    "tomato_leaf_miner": ["tomato leaf miner", "leaf miner tomato", "leafminer tomato"],
    "tomato_whitefly": ["tomato whitefly", "whitefly tomato"],
    "cucumber_healthy": ["cucumber healthy", "healthy cucumber"],
    "cucumber_downy_mildew": ["cucumber downy mildew", "downy mildew cucumber"],
    "cucumber_powdery_mildew": ["cucumber powdery mildew", "powdery mildew cucumber"],
    "cucumber_anthracnose": ["cucumber anthracnose", "anthracnose cucumber"],
    "cucumber_bacterial_wilt": ["cucumber bacterial wilt", "bacterial wilt cucumber"],
    "cucumber_belly_rot": ["cucumber belly rot", "belly rot cucumber"],
    "cucumber_pythium_fruit_rot": [
        "cucumber pythium fruit rot",
        "pythium fruit rot cucumber",
    ],
    "cucumber_gummy_stem_blight": [
        "cucumber gummy stem blight",
        "gummy stem blight cucumber",
    ],
    "pepper_healthy": [
        "pepper bell healthy",
        "pepper,_bell___healthy",
        "bell pepper healthy",
        "pepper healthy",
        "bell pepper leaf",
    ],
    "pepper_bacterial_spot": [
        "pepper bell bacterial spot",
        "pepper,_bell___bacterial_spot",
        "bell pepper bacterial spot",
        "pepper bacterial spot",
        "bell pepper leaf spot",
    ],
    "pepper_cercospora_leaf_spot": [
        "pepper cercospora leaf spot",
        "bell pepper cercospora leaf spot",
    ],
    "pepper_leaf_curl": ["pepper leaf curl", "bell pepper leaf curl"],
    "pepper_nutrient_deficiency": [
        "pepper nutrition deficiency",
        "pepper nutrient deficiency",
        "bell pepper nutrition deficiency",
    ],
    "pepper_powdery_mildew": ["pepper powdery mildew", "bell pepper powdery mildew"],
    "eggplant_healthy": ["eggplant healthy", "brinjal healthy"],
    "eggplant_insect_pest": ["eggplant insect pest", "brinjal insect pest"],
    "eggplant_leaf_spot": ["eggplant leaf spot", "brinjal leaf spot"],
    "eggplant_mosaic_virus": ["eggplant mosaic virus", "brinjal mosaic virus"],
    "eggplant_small_leaf": ["eggplant small leaf", "brinjal small leaf"],
    "eggplant_white_mold": ["eggplant white mold", "brinjal white mold"],
    "eggplant_wilt": ["eggplant wilt", "brinjal wilt"],
    "strawberry_healthy": [
        "strawberry healthy",
        "strawberry leaf",
        "healthy strawberry",
    ],
    "strawberry_leaf_scorch": ["strawberry leaf scorch", "strawberry scorch"],
    "strawberry_calcium_deficiency": [
        "strawberry calcium deficiency",
        "strawberry tipburn",
    ],
    "strawberry_gray_mold": ["strawberry gray mold", "gray mold strawberry"],
    "strawberry_leaf_spot": ["strawberry leaf spot", "leaf spot strawberry"],
    "strawberry_powdery_mildew_leaf": [
        "strawberry powdery mildew leaf",
        "powdery mildew leaf strawberry",
    ],
    "strawberry_anthracnose_fruit_rot": [
        "strawberry anthracnose fruit rot",
        "anthracnose fruit rot strawberry",
    ],
}


In [ ]:
def norm(s):
    return re.sub("\\s+", " ", re.sub("[^a-z0-9]+", " ", str(s).lower())).strip()


In [ ]:
LOOKUP = {norm(a): c for c, v in ALIASES.items() for a in v + [c]}


In [ ]:
def map_label(p, src):
    path_parts = [norm(part) for part in p.parts[-10:] if norm(part)]
    candidates = list(
        dict.fromkeys(
            [norm(p.parent.name), norm(p.parent.parent.name), norm(p.stem)]
            + list(reversed(path_parts))
        )
    )
    joined = " ".join(candidates)
    full_path = norm(str(p))
    if any(
        (
            x in joined
            for x in ["and leaf miner", "powdery mildew and", "bacterial spot and"]
        )
    ):
        return (None, "mixed")
    if src == "tlid":
        tlid = {
            "healthy": "tomato_healthy",
            "miner": "tomato_leaf_miner",
            "leaf miner": "tomato_leaf_miner",
            "miner fly": "tomato_leaf_miner",
            "bacterialspot": "tomato_bacterial_spot",
            "bacterial spot": "tomato_bacterial_spot",
            "powderymildew": "tomato_powdery_mildew",
            "powdery mildew": "tomato_powdery_mildew",
            "whitefly": "tomato_whitefly",
            "white fly": "tomato_whitefly",
        }
        numeric_tlid = {
            "0": "tomato_healthy",
            "1": "tomato_leaf_miner",
            "2": "tomato_bacterial_spot",
            "3": "tomato_powdery_mildew",
            "6": "tomato_whitefly",
        }
        for raw_part in reversed(p.parts[:-1]):
            match = re.match(
                "^(?:class|classe)?[ _-]*([0-6])(?:[ _-]|$)", str(raw_part).lower()
            )
            if match:
                class_id = match.group(1)
                return (
                    (numeric_tlid[class_id], "known")
                    if class_id in numeric_tlid
                    else (None, "mixed")
                )
        for x in candidates:
            for key, value in tlid.items():
                if x == key or x.endswith(" " + key) or key in x:
                    return (value, "known")
    if src == "olid_field":
        olid_class = next(
            (
                part
                for part in reversed(path_parts[:-1])
                if re.match("^(tomato|cucumber|eggplant|brinjal)(?: |$)", part)
            ),
            "",
        )
        tokens = olid_class.split()
        crop = tokens[0] if tokens else ""
        stress = tokens[1:]
        if crop and (not stress) and (norm(p.parent.name) not in {crop, "brinjal"}):
            stress = norm(p.parent.name).split()
        if crop == "tomato":
            if stress == ["healthy"]:
                return ("tomato_healthy", "known")
            if stress in [["lm"], ["leaf", "miner"], ["leafminer"]]:
                return ("tomato_leaf_miner", "known")
        if crop == "cucumber" and stress == ["healthy"]:
            return ("cucumber_healthy", "known")
        if crop in {"eggplant", "brinjal"}:
            if stress == ["healthy"]:
                return ("eggplant_healthy", "known")
            if len(stress) == 1 and stress[0] in {"eb", "fb", "jas", "mit"}:
                return ("eggplant_insect_pest", "known")
            if stress in [
                ["epilachna", "beetle"],
                ["flea", "beetle"],
                ["jassid"],
                ["mite"],
                ["spider", "mite"],
            ]:
                return ("eggplant_insect_pest", "known")
        if len(stress) > 1 and all(
            (item in {"eb", "fb", "jas", "mit"} for item in stress)
        ):
            return (None, "mixed")
        return (None, "olid_unmapped")
    if src == "eggplant_real_field":
        if "healthy" in full_path:
            return ("eggplant_healthy", "known")
        if "spider mite" in full_path or "mite" in candidates:
            return ("eggplant_insect_pest", "known")
        return (None, "coarse_unmapped")
    if src == "strawberry_field" and "tipburn" in full_path:
        tipburn_parent = norm(p.parent.name).replace(" ", "")
        if tipburn_parent in {"class0", "0", "calciumdeficiency"}:
            return ("strawberry_calcium_deficiency", "known")
        if tipburn_parent in {"class1", "1", "healthy"}:
            return ("strawberry_healthy", "known")
    crop_maps = {
        "cucumber_field": {
            "fresh leaves": "cucumber_healthy",
            "fresh cucumber": "cucumber_healthy",
            "anthracnose": "cucumber_anthracnose",
            "bacterial wilt": "cucumber_bacterial_wilt",
            "belly rot": "cucumber_belly_rot",
            "downy mildew": "cucumber_downy_mildew",
            "pythium fruit rot": "cucumber_pythium_fruit_rot",
            "gummy stem blight": "cucumber_gummy_stem_blight",
            "powdery mildew": "cucumber_powdery_mildew",
        },
        "pepper_field": {
            "healthy": "pepper_healthy",
            "bacterial spot": "pepper_bacterial_spot",
            "cercospora leaf spot": "pepper_cercospora_leaf_spot",
            "leaf curl": "pepper_leaf_curl",
            "curl virus": "pepper_leaf_curl",
            "nutrition deficiency": "pepper_nutrient_deficiency",
            "nutrient deficiency": "pepper_nutrient_deficiency",
            "powdery mildew": "pepper_powdery_mildew",
        },
        "eggplant_field": {
            "healthy": "eggplant_healthy",
            "healthy leaf": "eggplant_healthy",
            "insect pest": "eggplant_insect_pest",
            "insect pest disease": "eggplant_insect_pest",
            "leaf spot": "eggplant_leaf_spot",
            "leaf spot disease": "eggplant_leaf_spot",
            "mosaic virus": "eggplant_mosaic_virus",
            "mosaic virus disease": "eggplant_mosaic_virus",
            "small leaf": "eggplant_small_leaf",
            "small leaf disease": "eggplant_small_leaf",
            "white mold": "eggplant_white_mold",
            "white mold disease": "eggplant_white_mold",
            "wilt": "eggplant_wilt",
            "wilt disease": "eggplant_wilt",
        },
        "strawberry_field": {
            "healthy": "strawberry_healthy",
            "healthy strawberry": "strawberry_healthy",
            "leaf scorch": "strawberry_leaf_scorch",
            "calcium deficiency": "strawberry_calcium_deficiency",
            "tipburn": "strawberry_calcium_deficiency",
            "gray mold": "strawberry_gray_mold",
            "leaf spot": "strawberry_leaf_spot",
            "powdery mildew leaf": "strawberry_powdery_mildew_leaf",
            "anthracnose fruit rot": "strawberry_anthracnose_fruit_rot",
        },
    }
    for x in candidates:
        for key, value in crop_maps.get(src, {}).items():
            if x == key or x.endswith(" " + key) or key in x:
                return (value, "known")
    for x in candidates:
        if x in LOOKUP:
            return (LOOKUP[x], "known")
    hits = [(len(a), c) for a, c in LOOKUP.items() if a in joined]
    return (max(hits)[1], "known") if hits else (None, "unmapped")


In [ ]:
def source_name(p):
    s = norm(str(p))
    if "plantdoc" in s:
        parts = {norm(part) for part in p.parts}
        if "train" in parts:
            return "plantdoc_train"
        return "plantdoc_test"
    if "plantwild" in s or "plant wild" in s:
        return "plantwild"
    if "tlid" in s or "tomato leaf image" in s:
        return "tlid"
    if "plantvillage" in s or "plant village" in s:
        return "plantvillage"
    if "olid i" in s or "olid dataset" in s or "open field leaf image" in s:
        return "olid_field"
    if "cdd cucumber" in s or "cucumber disease" in s:
        return "cucumber_field"
    if (
        "pepper bell leaf disease" in s
        or "pepper disease" in s
        or "chilli leaf disease" in s
    ):
        return "pepper_field"
    if ("real field" in s or "spider mite" in s) and (
        "eggplant" in s or "brinjal" in s
    ):
        return "eggplant_real_field"
    if "eggplant" in s or "brinjal" in s:
        return "eggplant_field"
    if (
        "strawberry disease" in s
        or ("strawberry" in s and "tipburn" in s)
        or "strawberry coco" in s
    ):
        return "strawberry_field"
    if "hotbed field test" in s or "greenhouse holdout" in s:
        return "own_holdout"
    return "other"


In [ ]:
EXCLUDED_IDS = {
    52,
    53,
    55,
    56,
    57,
    64,
    94,
    97,
    119,
    122,
    124,
    129,
    134,
    142,
    153,
    163,
    169,
    170,
}

EXCLUDED_PIXELS = {value - 1 for value in EXCLUDED_IDS}

MASK_ARCHIVE = "stuffthingmaps_trainval2017.zip"

MASK_SHA256 = "da7f213d6a581bd6912a957573d7b4e5061de0d41f733bb27249b0fa981239d7"

MASK_URLS = (
    "https://calvin.inf.ed.ac.uk/wp-content/uploads/data/cocostuffdataset/"
    + MASK_ARCHIVE,
    "https://huggingface.co/hngan/my_lasagna/resolve/10465c12ad91d9637b25c8b32afadb2ddcc27e06/"
    + MASK_ARCHIVE,
)


In [ ]:
def validate_archive(path):
    with zipfile.ZipFile(path) as archive:
        for split in ("train2017", "val2017"):
            member = next(
                (
                    name
                    for name in archive.namelist()
                    if split in PurePosixPath(name).parts and name.endswith(".png")
                ),
                None,
            )
            if member is None:
                raise ValueError(f"Missing {split} PNG masks")
            with Image.open(io.BytesIO(archive.read(member))) as image:
                eligible_mask(np.asarray(image))


In [ ]:
def download_masks(config):
    target = Path(config["work_dir"]) / "downloads" / MASK_ARCHIVE
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        validate_archive(target)
        return target
    until = time.monotonic() + float(config.get("mask_download_minutes", 12)) * 60
    partial = target.with_suffix(".zip.part")
    errors = []
    for url in MASK_URLS:
        if time.monotonic() >= until:
            break
        print("MASK DOWNLOAD:", url, flush=True)
        try:
            request = urllib.request.Request(
                url, headers={"User-Agent": "HotBed-dataset-preparation/1.0"}
            )
            with urllib.request.urlopen(
                request, timeout=min(30.0, max(1.0, until - time.monotonic()))
            ) as response:
                digest, written, last = (hashlib.sha256(), 0, time.monotonic())
                with partial.open("wb") as output:
                    while True:
                        if time.monotonic() >= until:
                            raise TimeoutError('Mask download timed out')
                        block = response.read(1024 * 1024)
                        if not block:
                            break
                        output.write(block)
                        digest.update(block)
                        written += len(block)
                        if time.monotonic() - last >= 15:
                            print(
                                f"MASK DOWNLOAD: {written / 1000000.0:.1f} / 659.0 MB",
                                flush=True,
                            )
                            last = time.monotonic()
            if digest.hexdigest() != MASK_SHA256:
                raise ValueError('Mask checksum mismatch')
            validate_archive(partial)
            partial.replace(target)
            target.with_suffix(".provenance.json").write_text(
                json.dumps(dict(url=url, sha256=MASK_SHA256, bytes=written), indent=2),
                encoding="utf-8",
            )
            return target
        except (OSError, ValueError, zipfile.BadZipFile) as error:
            errors.append(f"{type(error).__name__}: {error}")
            print("MASK DOWNLOAD failed:", errors[-1], flush=True)
            partial.unlink(missing_ok=True)
    raise ValueError(
        "COCO-Stuff download failed"
    )


In [ ]:
def ensure_coco_inputs(config, paths):
    """Check required inputs before weights, image hashing, or annotation crops."""
    if not config.get("coco_stuff_negatives", False):
        return paths
    for split in ("train2017", "val2017"):
        first = next(
            (
                p
                for p in paths
                if p.suffix.lower() in {".jpg", ".jpeg"}
                and p.stem.isdigit()
                and (split in p.parts)
            ),
            None,
        )
        if first is None:
            raise ValueError(
                f"COCO {split} images not found"
            )
        print(f"COCO {split}: found", flush=True)
    archives = [p for p in paths if p.name == MASK_ARCHIVE]
    masks = [
        p
        for p in paths
        if p.suffix == ".png"
        and p.stem.isdigit()
        and any((s in p.parts for s in ("train2017", "val2017")))
    ]
    if len(archives) > 1:
        raise ValueError("More than one COCO-Stuff archive mounted")
    if not archives and (not masks):
        if not config.get("auto_download_coco_masks", False):
            raise ValueError(
                f"Missing {MASK_ARCHIVE}"
            )
        paths = list(paths) + [download_masks(config)]
        archives = [paths[-1]]
    if archives:
        validate_archive(archives[0])
    else:
        for split in ("train2017", "val2017"):
            first = next((p for p in masks if split in p.parts), None)
            if first is None:
                raise ValueError(f"Missing {split} original COCO-Stuff PNG masks")
            with Image.open(first) as image:
                eligible_mask(np.asarray(image))
    return paths


In [ ]:
def eligible_mask(mask):
    if mask.ndim != 2 or not np.isin(mask, list(range(182)) + [255]).all():
        raise ValueError(
            'Invalid COCO-Stuff mask format'
        )
    if np.isin(mask, list(EXCLUDED_PIXELS)).any():
        return False
    return float(np.mean(mask == 255)) <= 0.005


In [ ]:
def coco_negatives(config, paths):
    if not config.get("coco_stuff_negatives", False):
        return []
    archives = [p for p in paths if p.name == "stuffthingmaps_trainval2017.zip"]
    masks = [
        p
        for p in paths
        if p.suffix == ".png"
        and p.stem.isdigit()
        and any((s in p.parts for s in ("train2017", "val2017")))
    ]
    if len(archives) > 1 or (not archives and (not masks)):
        raise ValueError(
            'COCO-Stuff masks not found'
        )
    images = {}
    for path in paths:
        if path.suffix.lower() not in {".jpg", ".jpeg"} or not path.stem.isdigit():
            continue
        split = next((s for s in ("train2017", "val2017") if s in path.parts), None)
        if split:
            key = (split, path.stem)
            if key in images:
                raise ValueError(f"Duplicate mounted COCO image: {key}")
            images[key] = path
    if not all(
        (any((key[0] == split for key in images)) for split in ("train2017", "val2017"))
    ):
        raise ValueError(
            'COCO train2017/val2017 not found'
        )
    targets = config.get(
        "coco_negative_counts", dict(train=4000, calibration=800, field_test=800)
    )
    rng = random.Random(int(config["seed"]))
    count, rejected, rows = (Counter(), Counter(), [])
    until = time.monotonic() + float(config.get("negative_prepare_minutes", 10)) * 60
    with zipfile.ZipFile(archives[0]) if archives else nullcontext(None) as archive:
        by_split = {"train2017": [], "val2017": []}
        for member in archive.namelist() if archive else [p.as_posix() for p in masks]:
            p = PurePosixPath(member)
            split = next((s for s in by_split if s in p.parts), None)
            if split and p.suffix == ".png" and ((split, p.stem) in images):
                by_split[split].append(member)
        for split, members in by_split.items():
            print(f"NEGATIVES: {split}, {len(members)} matching masks", flush=True)
            if not members:
                raise ValueError(f"No masks match COCO images in {split}")
            split_until = (
                time.monotonic() + 0.8 * max(0.0, until - time.monotonic())
                if split == "train2017"
                else until
            )
            rng.shuffle(members)
            destinations = (
                ["train"] * targets["train"] + ["calibration"] * targets["calibration"]
                if split == "train2017"
                else ["field_test"] * targets["field_test"]
            )
            rng.shuffle(destinations)
            accepted = 0
            last_progress = time.monotonic()
            for scanned, member in enumerate(members, 1):
                if time.monotonic() - last_progress >= 15:
                    print(f"{split}: checked {scanned}, selected {accepted}/{len(destinations)}", flush=True)
                    last_progress = time.monotonic()
                if accepted == len(destinations) or time.monotonic() > split_until:
                    break
                with Image.open(
                    io.BytesIO(archive.read(member)) if archive else member
                ) as image:
                    mask = np.asarray(image)
                if not eligible_mask(mask):
                    rejected["plant_produce_or_void"] += 1
                    continue
                image_id = PurePosixPath(member).stem
                path = images[split, image_id]
                with Image.open(path) as photo:
                    if photo.size != (mask.shape[1], mask.shape[0]):
                        raise ValueError(f"Image/mask size mismatch: {path.name}")
                destination = destinations[accepted]
                rows.append(
                    dict(
                        path=str(path),
                        label="nonplant",
                        source="coco_stuff_negatives",
                        origin=f"coco2017:{image_id}",
                        requested_split=destination,
                    )
                )
                count[destination] += 1
                accepted += 1
    report = dict(
        selected=dict(count),
        excluded=dict(rejected),
        targets=targets,
        scope="Original COCO-Stuff annotations; residual annotation errors remain possible",
        field_test_source="COCO val2017; train/calibration use COCO train2017",
    )
    Path(config["work_dir"], "negative_selection.json").write_text(
        json.dumps(report, indent=2)
    )
    print("Nonplant samples:", dict(count), flush=True)
    if any(
        (
            count[split] < int(config.get("minimum_negative_per_split", 50))
            for split in ("train", "calibration", "field_test")
        )
    ):
        raise ValueError(
            'Too few COCO negatives'
        )
    return rows


In [ ]:
EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

HOLDOUT_SOURCES = {"plantdoc_test", "own_holdout"}

SPLITS = ("train", "val", "calibration", "field_test", "domain_test")


In [ ]:
def sha_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


In [ ]:
def input_paths(config):
    """One filesystem walk; avoid a stat/resolve call for every COCO image."""
    root = Path(config["input_root"])
    if not root.is_dir():
        raise FileNotFoundError(root)
    print("INPUT: listing datasets under", root, flush=True)
    paths, last = ([], time.monotonic())
    for directory, folders, files in os.walk(root):
        folders[:] = [
            name for name in folders if name not in {"test2017", ".git", "__pycache__"}
        ]
        paths.extend((Path(directory) / name for name in files))
        if time.monotonic() - last >= 15:
            print(f"Files found: {len(paths)}", flush=True)
            last = time.monotonic()
    print(f"INPUT: found {len(paths)} files; COCO test2017 excluded", flush=True)
    return sorted(paths)


In [ ]:
class UnionFind:

    def __init__(self, n):
        self.parents = list(range(n))

    def find(self, x):
        while x != self.parents[x]:
            self.parents[x] = self.parents[self.parents[x]]
            x = self.parents[x]
        return x

    def union(self, a, b):
        self.parents[self.find(b)] = self.find(a)


In [ ]:
def group_images(frame, near_bits=4):
    """Group near duplicates globally, then join crops sharing an original.

    Conflicting labels on duplicate images are quarantined, not silently chosen.
    Different objects from the same original may legitimately have different labels.
    """
    frame = frame.reset_index(drop=True).copy()
    uf = UnionFind(len(frame))
    exact, buckets = ({}, {})
    bands = near_bits + 1
    boundaries = np.linspace(0, 64, bands + 1, dtype=int)
    hashes = [int(value, 16) for value in frame.phash]
    last_progress = time.monotonic()
    hash_representative = {}
    for i, row in frame.iterrows():
        if time.monotonic() - last_progress >= 15:
            print(f"DEDUP: {i}/{len(frame)} images", flush=True)
            last_progress = time.monotonic()
        for key in (("sha", row.sha256), ("pixels", row.pixel_sha256)):
            if key in exact:
                uf.union(i, exact[key])
            exact[key] = i
        if hashes[i] in hash_representative:
            uf.union(i, hash_representative[hashes[i]])
            continue
        hash_representative[hashes[i]] = i
        candidates = set()
        keys = []
        for b in range(bands):
            width = int(boundaries[b + 1] - boundaries[b])
            key = (b, hashes[i] >> int(boundaries[b]) & (1 << width) - 1)
            keys.append(key)
            candidates.update(buckets.get(key, ()))
        for j in candidates:
            if (hashes[i] ^ hashes[j]).bit_count() <= near_bits:
                uf.union(i, j)
        for key in keys:
            buckets.setdefault(key, []).append(i)
    frame["duplicate_group"] = [uf.find(i) for i in range(len(frame))]
    conflicts = set(
        frame.groupby("duplicate_group").label.nunique().loc[lambda s: s > 1].index
    )
    rejected = frame[frame.duplicate_group.isin(conflicts)].copy()
    rejected["reason"] = "duplicate_label_conflict"
    originals = {}
    for i, original in enumerate(frame.origin):
        if original in originals:
            uf.union(i, originals[original])
        originals[original] = i
    frame["group"] = [uf.find(i) for i in range(len(frame))]
    frame = frame[~frame.duplicate_group.isin(conflicts)].copy()
    priority = {
        "": -1,
        "train": 0,
        "val": 1,
        "calibration": 2,
        "field_test": 3,
        "domain_test": 4,
    }
    for _, part in frame.groupby("group"):
        requested = set(part.requested_split.fillna(""))
        if part.source.isin(HOLDOUT_SOURCES).any():
            requested.add("domain_test")
        frame.loc[part.index, "requested_split"] = max(
            requested, key=priority.__getitem__
        )
    frame["_holdout"] = frame.source.isin(HOLDOUT_SOURCES) | frame.requested_split.isin(
        ["field_test", "domain_test"]
    )
    frame = frame.sort_values("_holdout", ascending=False).drop_duplicates(
        "pixel_sha256"
    )
    return (frame.drop(columns="_holdout").reset_index(drop=True), rejected)


In [ ]:
def assign_splits(frame, seed=42):
    """One split per global group; explicit calibration/test constraints are retained."""
    frame = frame.copy()
    group_split = {}
    priority = {
        "train": 0,
        "val": 1,
        "calibration": 2,
        "field_test": 3,
        "domain_test": 4,
    }
    for group, part in frame.groupby("group"):
        requested = set(part.requested_split.dropna()) - {""}
        if part.source.isin(HOLDOUT_SOURCES).any():
            requested.add("domain_test")
        if requested:
            group_split[group] = max(requested, key=priority.__getitem__)
    representative = frame.drop_duplicates("group")
    rng = np.random.default_rng(seed)
    for _, part in representative.groupby(["source", "label"], sort=True):
        groups = [g for g in part.group if g not in group_split]
        rng.shuffle(groups)
        n = len(groups)
        count = max(1, round(n * 0.1)) if n >= 10 else 0
        for j, group in enumerate(groups):
            group_split[group] = (
                "field_test"
                if j < count
                else (
                    "calibration"
                    if j < 2 * count
                    else "val" if j < 3 * count else "train"
                )
            )
    frame["split"] = frame.group.map(group_split)
    assert frame.groupby("group").split.nunique().max() == 1
    return frame


In [ ]:
def scan(config, paths=None):
    import imagehash

    root, work = (Path(config["input_root"]), Path(config["work_dir"]))
    work.mkdir(parents=True, exist_ok=True)
    paths = input_paths(config) if paths is None else paths
    paths = ensure_coco_inputs(config, paths)
    negative_rows = coco_negatives(config, paths)
    crops_dir = work / "annotation_crops"
    crops_dir.mkdir(exist_ok=True)
    records, rejected, annotated_originals = ([], [], set())
    last_progress = time.monotonic()

    def add(path, label, source, origin=None, requested_split=""):
        nonlocal last_progress
        try:
            with Image.open(path) as image:
                image = ImageOps.exif_transpose(image).convert("RGB")
                if min(image.size) < 32:
                    raise ValueError("minimum dimension < 32")
                pixel_sha = hashlib.sha256(
                    str(image.size).encode() + image.tobytes()
                ).hexdigest()
                phash = str(imagehash.phash(image))
            digest = sha_file(path)
            records.append(
                dict(
                    path=str(path),
                    label=label,
                    source=source,
                    origin=origin or digest,
                    sha256=digest,
                    pixel_sha256=pixel_sha,
                    phash=phash,
                    requested_split=requested_split,
                )
            )
        except (OSError, ValueError) as error:
            rejected.append(dict(path=str(path), reason=f"broken:{error}"))
        if time.monotonic() - last_progress >= 15:
            print(
                f"IMAGES: indexed={len(records)} rejected={len(rejected)} source={source}",
                flush=True,
            )
            last_progress = time.monotonic()

    def crop_object(original, label, box, key):
        annotated_originals.add(original.resolve())
        with Image.open(original) as image:
            image = image.convert("RGB")
            x0, y0, x1, y1 = box
            w, h = (x1 - x0, y1 - y0)
            x0, y0 = (max(0, int(x0 - w * 0.12)), max(0, int(y0 - h * 0.12)))
            x1, y1 = (
                min(image.width, int(x1 + w * 0.12)),
                min(image.height, int(y1 + h * 0.12)),
            )
            if min(x1 - x0, y1 - y0) < 32:
                return
            target = crops_dir / (hashlib.sha256(key.encode()).hexdigest() + ".jpg")
            image.crop((x0, y0, x1, y1)).save(target, quality=95)
        add(target, label, "strawberry_field", origin=sha_file(original))

    name_cache = {}

    def resolve_image(directory, name):
        direct = directory / name
        if direct.is_file():
            return direct
        if directory not in name_cache:
            table = {}
            for p in directory.rglob("*"):
                if p.suffix.lower() in EXTENSIONS:
                    table.setdefault(p.name, []).append(p)
            name_cache[directory] = table
        matches = name_cache[directory].get(Path(name).name, [])
        if len(matches) > 1:
            raise ValueError(f"Ambiguous image: {name}")
        return matches[0] if matches else None

    for path in paths:
        if path.suffix.lower() != ".json" or "strawberry" not in norm(path):
            continue
        if any(
            (
                norm(p) in {"augmentation", "augmentations", "synthetic"}
                for p in path.parts
            )
        ):
            continue
        try:
            doc = json.loads(path.read_text(encoding="utf-8"))
        except (ValueError, UnicodeError):
            continue
        if not isinstance(doc, dict):
            continue
        if "annotations" in doc and "categories" in doc:
            categories = {x["id"]: x["name"] for x in doc["categories"]}
            images = {x["id"]: x for x in doc.get("images", [])}
            for ann in doc["annotations"]:
                label, _ = map_label(
                    Path(categories.get(ann["category_id"], "")) / "sample.jpg",
                    "strawberry_field",
                )
                info = images.get(ann["image_id"])
                if label is None or not info or (not ann.get("bbox")):
                    continue
                original = resolve_image(path.parent, info["file_name"])
                if original:
                    x, y, w, h = ann["bbox"]
                    crop_object(
                        original, label, (x, y, x + w, y + h), f"{path}:{ann['id']}"
                    )
        elif doc.get("shapes") and doc.get("imagePath"):
            original = resolve_image(path.parent, doc["imagePath"])
            if original:
                for j, shape in enumerate(doc["shapes"]):
                    label, _ = map_label(
                        Path(shape.get("label", "")) / "sample.jpg", "strawberry_field"
                    )
                    points = shape.get("points", [])
                    if label is not None and len(points) >= 2:
                        xs, ys = zip(*points)
                        crop_object(
                            original,
                            label,
                            (min(xs), min(ys), max(xs), max(ys)),
                            f"{path}:{j}",
                        )
    for p in paths:
        if p.suffix.lower() not in EXTENSIONS:
            continue
        src = source_name(p)
        if src == "other":
            continue
        if p.resolve() in annotated_originals:
            continue
        parts = {norm(part) for part in p.parts}
        if parts & {
            "augmented image",
            "augmented images",
            "augmentation",
            "augmentations",
            "synthetic",
        }:
            continue
        if "abdallahalidev plantvillage dataset" in norm(p):
            continue
        import re

        if src == "tlid" and (
            "ptlid" in parts
            or any((re.match("^[0-9]+ patch(?: |$)", v) for v in parts))
        ):
            continue
        label, reason = map_label(p, src)
        if label is None:
            rejected.append(dict(path=str(p), source=src, reason=reason))
        else:
            add(p, label, src)
    for row in negative_rows:
        add(**row)
    manifests = (
        [p for p in paths if p.name == "hotbed_extra_manifest.csv"]
        if config.get("extra_manifest") == "auto"
        else [Path(config["extra_manifest"])] if config.get("extra_manifest") else []
    )
    for manifest in manifests:
        extra = pd.read_csv(manifest).fillna("")
        if not {"path", "label", "source", "group_id", "split"} <= set(extra):
            raise ValueError('Missing manifest columns')
        for row in extra.to_dict("records"):
            if row["label"] not in CLASSES + ["nonplant"] or row[
                "split"
            ] not in SPLITS + ("",):
                raise ValueError("Invalid manifest label or split")
            if not row["source"] or not row["group_id"]:
                raise ValueError('Empty source or group_id')
            p = Path(row["path"])
            p = p if p.is_absolute() else manifest.parent / p
            if not p.is_file():
                raise FileNotFoundError(p)
            add(
                p,
                row["label"],
                row["source"],
                origin=f"extra:{row['source']}:{row['group_id']}",
                requested_split=row["split"],
            )
    if not records:
        raise ValueError("No mapped images found under /kaggle/input")
    print(f"Deduplication: {len(records)} images", flush=True)
    frame, conflicts = group_images(
        pd.DataFrame(records), int(config["near_duplicate_bits"])
    )
    frame = assign_splits(frame, int(config["seed"]))
    keep = []
    for (split, source, label), part in frame.groupby(["split", "source", "label"]):
        cap = (
            (
                config["plantvillage_train_cap"]
                if source == "plantvillage"
                else config["train_cap_per_class_source"]
            )
            if split == "train"
            else len(part)
        )
        if label == "nonplant" and source == "coco_stuff_negatives":
            cap = len(part)
        keep.extend(
            part.sample(n=min(int(cap), len(part)), random_state=config["seed"]).index
        )
    frame = frame.loc[sorted(keep)].reset_index(drop=True)
    frame["row_id"] = np.arange(len(frame))
    missing = sorted(set(CLASSES) - set(frame.loc[frame.split == "train", "label"]))
    if missing:
        raise ValueError(
            f"Missing training classes: {len(missing)}"
        )
    for split in ["val", "calibration", "field_test", "domain_test"]:
        if not ((frame.split == split) & (frame.label != "nonplant")).any():
            raise ValueError(f"No plant examples for {split}")
    frame.to_csv(work / "split_manifest.csv", index=False)
    pd.concat([pd.DataFrame(rejected), conflicts], ignore_index=True).to_csv(
        work / "rejected_images.csv", index=False
    )
    report = (
        frame.groupby(["split", "source", "label"])
        .size()
        .rename("samples")
        .reset_index()
    )
    report.to_csv(work / "split_counts.csv", index=False)
    print(report.to_string(index=False))
    for split in SPLITS:
        support = set(
            frame.loc[(frame.split == split) & (frame.label != "nonplant"), "label"]
        )
        print(split, "missing classes:\n" + "\n".join(sorted(set(CLASSES) - support)))
    print("PREPARE DONE:", work / "split_manifest.csv", flush=True)
    return frame


## 5. Модель и загрузчик изображений

In [ ]:
CROPS = ("tomato", "cucumber", "pepper", "eggplant", "strawberry")
CLASS_CROP = torch.tensor([CROPS.index(name.split("_")[0]) for name in CLASSES])
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
BACKBONES = ("vit_base_patch14_dinov2.lvd142m", "vit_base_patch16_dinov3.lvd1689m")


In [ ]:
def active_candidates(config):
    return [
        c
        for c in config["candidates"]
        if not c.get("enabled_by") or config.get(c["enabled_by"], False)
    ]


In [ ]:
def preflight_backbones(config):
    """Resolve pretrained weights once, before starting expensive distributed training."""
    import timm

    checked = set()
    for candidate in active_candidates(config):
        name = candidate.get("backbone", config["backbone"])
        if name in checked:
            continue
        weight_file = config.get("dinov3_pretrained_file") if "dinov3" in name else None
        if weight_file and (not Path(weight_file).is_file()):
            raise FileNotFoundError(weight_file)
        options = (
            {"pretrained_cfg_overlay": {"file": str(weight_file)}}
            if weight_file
            else {}
        )
        model = timm.create_model(
            name, pretrained=True, num_classes=0, img_size=candidate["imgsz"], **options
        )
        del model
        checked.add(name)


In [ ]:
def atomic_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False),
        encoding="utf-8",
    )
    temporary.replace(path)


In [ ]:
def atomic_save(path, value):
    path = Path(path)
    temporary = path.with_suffix(".tmp")
    torch.save(value, temporary)
    temporary.replace(path)


In [ ]:
def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


In [ ]:
def crop_loss(logits, labels, smoothing=0.04):
    """Smooth over the selected crop ONLY; -1 nonplant labels have no disease loss."""
    losses = []
    crop_map = CLASS_CROP.to(logits.device)
    for crop in range(len(CROPS)):
        columns = torch.where(crop_map == crop)[0]
        rows = (labels >= 0) & (crop_map[labels.clamp_min(0)] == crop)
        if rows.any():
            targets = (labels[rows, None] == columns[None, :]).long().argmax(1)
            losses.append(
                F.cross_entropy(
                    logits[rows][:, columns].float(),
                    targets,
                    label_smoothing=smoothing,
                    reduction="sum",
                )
            )
    return sum(losses, logits.sum() * 0.0) / (labels >= 0).sum().clamp_min(1)


In [ ]:
def conditional_predictions(logits, labels):
    allowed = (
        CLASS_CROP.numpy()[None, :] == CLASS_CROP.numpy()[np.maximum(labels, 0), None]
    )
    return np.where(allowed, logits, -1000000000.0).argmax(1)


In [ ]:
class FieldModel(nn.Module):
    """Disease classifier with an independent plantness head."""

    def __init__(
        self,
        imgsz=392,
        pretrained=False,
        backbone=None,
        backbone_name=BACKBONES[0],
        pretrained_file=None,
    ):
        super().__init__()
        if backbone is None:
            import timm

            if backbone_name not in BACKBONES:
                raise ValueError(f"Unsupported backbone: {backbone_name}")
            options = (
                {"pretrained_cfg_overlay": {"file": str(pretrained_file)}}
                if pretrained_file
                else {}
            )
            backbone = timm.create_model(
                backbone_name,
                pretrained=pretrained,
                num_classes=0,
                img_size=imgsz,
                **options,
            )
        self.backbone_name = backbone_name
        self.backbone = backbone
        dim = int(backbone.embed_dim)
        self.patch_gate = nn.Sequential(
            nn.Linear(dim, max(64, dim // 4)),
            nn.GELU(),
            nn.Linear(max(64, dim // 4), 1),
        )
        self.project = nn.Sequential(
            nn.LayerNorm(dim * 2),
            nn.Linear(dim * 2, 384),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.LayerNorm(384),
        )
        self.linear = nn.Linear(384, len(CLASSES), bias=False)
        self.prototypes = nn.Parameter(torch.randn(len(CLASSES), 384) * 0.02)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)))
        self.plant_head = nn.Sequential(
            nn.LayerNorm(dim * 2), nn.Linear(dim * 2, 128), nn.GELU(), nn.Linear(128, 1)
        )

    def forward(self, images):
        tokens = self.backbone.forward_features(images)
        if isinstance(tokens, dict):
            cls, patches = (tokens["x_norm_clstoken"], tokens["x_norm_patchtokens"])
        else:
            cls = tokens[:, 0]
            patches = tokens[:, self.backbone.num_prefix_tokens :]
        attention = self.patch_gate(patches).squeeze(-1).softmax(-1)
        local = (patches * attention.unsqueeze(-1)).sum(1)
        pooled = torch.cat((cls, local), dim=1)
        embedding = F.normalize(self.project(pooled), dim=1)
        scale = self.logit_scale.exp().clamp(max=100)
        logits = (
            0.5
            * scale
            * (
                F.linear(embedding, F.normalize(self.linear.weight, dim=1))
                + embedding @ F.normalize(self.prototypes, dim=1).T
            )
        )
        return (logits, embedding, self.plant_head(pooled))


In [ ]:
def load_initial(model, path):
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    if checkpoint["classes"] != CLASSES:
        raise ValueError("Checkpoint taxonomy/order mismatch")
    if checkpoint.get("backbone", model.backbone_name) != model.backbone_name:
        raise ValueError('Checkpoint backbone mismatch')
    state = dict(checkpoint["state_dict"])
    key = "backbone.pos_embed"
    if (
        key in state
        and key in model.state_dict()
        and (state[key].shape != model.state_dict()[key].shape)
    ):
        prefix = model.backbone.num_prefix_tokens
        old = state[key]
        side = math.isqrt(old.shape[1] - prefix)
        new_side = math.isqrt(model.state_dict()[key].shape[1] - prefix)
        assert side * side == old.shape[1] - prefix
        grid = old[:, prefix:].reshape(1, side, side, -1).permute(0, 3, 1, 2)
        grid = (
            F.interpolate(
                grid,
                (new_side, new_side),
                mode="bicubic",
                align_corners=False,
                antialias=True,
            )
            .permute(0, 2, 3, 1)
            .reshape(1, new_side * new_side, -1)
        )
        state[key] = torch.cat((old[:, :prefix], grid), dim=1)
    missing, unexpected = model.load_state_dict(state, strict=False)
    if unexpected or any((not name.startswith("plant_head.") for name in missing)):
        raise ValueError(
            "Checkpoint layer mismatch"
        )


In [ ]:
class CameraVariation:

    def __call__(self, image):
        import io

        if random.random() < 0.2:
            image = image.filter(ImageFilter.GaussianBlur(random.uniform(0.1, 0.7)))
        if random.random() < 0.2:
            buffer = io.BytesIO()
            image.save(buffer, format="JPEG", quality=random.randint(65, 95))
            buffer.seek(0)
            image = Image.open(buffer).convert("RGB")
        return image


In [ ]:
class Images(Dataset):

    def __init__(self, frame, size, train=False):
        self.frame = frame.reset_index(drop=True)
        self.size = size
        self.train = train
        self.augment = T.Compose(
            [
                T.RandomHorizontalFlip(),
                T.RandomVerticalFlip(0.1),
                T.RandomApply([T.ColorJitter(0.18, 0.18, 0.12, 0.015)], p=0.65),
                T.RandomRotation(12),
                CameraVariation(),
                T.ToTensor(),
                T.Normalize(MEAN, STD),
            ]
        )
        self.eval_transform = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row.path) as original:
            image = ImageOps.exif_transpose(original).convert("RGB")
        if self.train:
            global_symptom = any(
                (
                    v in row.label
                    for v in [
                        "healthy",
                        "wilt",
                        "curl",
                        "mosaic",
                        "nutrient",
                        "small_leaf",
                    ]
                )
            )
            image = T.RandomResizedCrop(
                self.size,
                scale=(0.8 if global_symptom else 0.65, 1.0),
                ratio=(0.85, 1.18),
                interpolation=T.InterpolationMode.BICUBIC,
            )(image)
            tensor = self.augment(image)
        else:
            image = image.resize((self.size, self.size), Image.Resampling.BICUBIC)
            tensor = self.eval_transform(image)
        label = CLASSES.index(row.label) if row.label in CLASSES else -1
        return (tensor, label, int(row.row_id))


In [ ]:
def balanced_weights(frame):
    labels = frame.groupby("label").size().to_dict()
    groups = frame.groupby(["label", "source"]).size().to_dict()
    values = np.asarray(
        [
            1
            / math.sqrt(labels[r.label] * groups[r.label, r.source])
            * (0.4 if r.source == "plantvillage" else 1.0)
            for r in frame.itertuples()
        ],
        dtype=np.float64,
    )
    negative = frame.label.to_numpy() == "nonplant"
    if negative.any():
        values[negative] *= 0.2 / values[negative].sum()
        values[~negative] *= 0.8 / values[~negative].sum()
    return values / values.sum()


In [ ]:
class BalancedDistributedSampler(Sampler):

    def __init__(self, weights, batch_size, rank, world, seed, max_samples=None):
        self.weights = torch.as_tensor(weights, dtype=torch.double)
        self.rank, self.world, self.seed, self.epoch = (rank, world, seed, 0)
        count = min(len(weights), max_samples) if max_samples else len(weights)
        self.total = math.ceil(count / (batch_size * world)) * batch_size * world

    def __len__(self):
        return self.total // self.world

    def __iter__(self):
        generator = torch.Generator().manual_seed(self.seed + self.epoch)
        samples = torch.multinomial(
            self.weights, self.total, replacement=True, generator=generator
        )
        return iter(samples[self.rank :: self.world].tolist())


In [ ]:
class NotebookDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        return self.dataset[index]

    def __reduce__(self):
        return cloudpickle.loads, (cloudpickle.dumps(self.dataset),)


In [ ]:
def loader(frame, size, workers, batch, train=False, sampler=None):
    return DataLoader(
        NotebookDataset(Images(frame, size, train)),
        batch_size=batch,
        sampler=sampler,
        shuffle=False,
        num_workers=workers,
        multiprocessing_context="spawn" if workers > 0 else None,
        pin_memory=True,
        persistent_workers=workers > 0,
    )


## 6. Обучение и валидация

In [ ]:
@torch.no_grad()
def infer(model, frame, size, workers, batch, device):
    model.eval()
    ids, labels, logits, embeddings, plants = ([], [], [], [], [])
    for images, target, row_ids in loader(frame, size, workers, batch):
        output = model(images.to(device))
        ids.extend(row_ids.tolist())
        labels.extend(target.tolist())
        logits.append(output[0].float().cpu().numpy())
        embeddings.append(output[1].float().cpu().numpy())
        plants.append(output[2].float().cpu().numpy().reshape(-1))
    if not ids:
        return dict(
            ids=np.empty(0, dtype=int),
            labels=np.empty(0, dtype=int),
            logits=np.empty((0, 39)),
            embeddings=np.empty((0, 384)),
            plants=np.empty(0),
        )
    return dict(
        ids=np.array(ids),
        labels=np.array(labels),
        logits=np.concatenate(logits),
        embeddings=np.concatenate(embeddings),
        plants=np.concatenate(plants),
    )


In [ ]:
def classification_metrics(labels, predictions):
    from sklearn.metrics import f1_score, balanced_accuracy_score

    keep = labels >= 0
    labels, predictions = (labels[keep], predictions[keep])
    supported = sorted(np.unique(labels).tolist())
    if not len(labels):
        return {}
    per_class = f1_score(
        labels, predictions, labels=supported, average=None, zero_division=0
    )
    return dict(
        samples=len(labels),
        supported_macro_f1=float(np.mean(per_class)),
        accuracy=float(np.mean(labels == predictions)),
        balanced_accuracy=float(balanced_accuracy_score(labels, predictions)),
        per_class_f1={CLASSES[c]: float(v) for c, v in zip(supported, per_class)},
        supported_classes=[CLASSES[c] for c in supported],
    )


In [ ]:
def selection_metrics(data, frame):
    prediction = conditional_predictions(data["logits"], data["labels"])
    overall = classification_metrics(data["labels"], prediction)
    rows = frame.set_index("row_id").loc[data["ids"]]
    source_scores = []
    for source in sorted(set(rows.source)):
        keep = (rows.source.to_numpy() == source) & (data["labels"] >= 0)
        if keep.any():
            source_scores.append(
                classification_metrics(data["labels"][keep], prediction[keep])[
                    "supported_macro_f1"
                ]
            )
    overall["source_macro_f1"] = float(np.mean(source_scores))
    overall["selection_score"] = (
        0.75 * overall["supported_macro_f1"] + 0.25 * overall["source_macro_f1"]
    )
    return overall


In [ ]:
def run_fingerprint(config, candidate):
    payload = (
        sha_file(Path(config["work_dir"]) / "split_manifest.csv")
        + json.dumps(config, sort_keys=True)
        + json.dumps(candidate, sort_keys=True)
        + NOTEBOOK_DIGEST
    )
    return hashlib.sha256(payload.encode()).hexdigest()


In [ ]:
def deadlines(config, started, now):
    total = float(config["session_budget_hours"]) * 3600
    reserve = float(config["finalization_reserve_hours"]) * 3600
    if not 0 < reserve < total:
        raise ValueError(
            'Invalid finalization time reserve'
        )
    end = started + total
    train_end = end - reserve
    return (end, train_end, max(0.0, train_end - now))


In [ ]:
class BudgetLR:
    """Finish cosine decay even when wall time permits fewer than the maximum epochs."""

    def __init__(self, updates, started, deadline):
        self.updates = max(1, updates)
        self.started = started
        self.deadline = deadline

    def __call__(self, step):
        import math

        elapsed = max(0.0, time.time() - self.started)
        duration = max(1.0, self.deadline - self.started)
        progress = min(1.0, max(step / self.updates, elapsed / duration))
        warmup = min(1.0, (step + 1) / min(100, max(1, self.updates // 12)))
        return warmup * (0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * progress)))


In [ ]:
def train(config, candidate):
    rank, world = (int(os.environ.get("RANK", 0)), int(os.environ.get("WORLD_SIZE", 1)))
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)
    dist.init_process_group("nccl", timeout=timedelta(hours=2))
    seed_all(candidate["seed"] + rank)
    work = Path(config["work_dir"])
    output = work / candidate["name"]
    output.mkdir(exist_ok=True)
    frame = pd.read_csv(work / "split_manifest.csv").fillna("")
    fingerprint = run_fingerprint(config, candidate)
    if (output / "done.json").exists():
        saved = json.loads((output / "done.json").read_text())
        if saved["fingerprint"] != fingerprint:
            raise ValueError(
                'Saved run/config mismatch'
            )
        dist.destroy_process_group()
        return
    backbone_name = candidate.get("backbone", config["backbone"])
    initial = None
    if candidate["init"] == "baseline":
        raise ValueError(
            'Use pretrained initialization'
        )
    if candidate["init"] == "best_previous":
        previous_names = []
        for item in active_candidates(config):
            if item["name"] == candidate["name"]:
                break
            if item.get("backbone", config["backbone"]) == backbone_name:
                previous_names.append(item["name"])
        earlier = [
            json.loads((work / name / "done.json").read_text())
            for name in previous_names
            if (work / name / "done.json").exists()
        ]
        if not earlier:
            raise ValueError(
                'Initial checkpoint not found'
            )
        best = max(earlier, key=lambda item: item["selection_score"])
        initial = work / best["name"] / "best.pth"
    pretrained_file = (
        config.get("dinov3_pretrained_file") if "dinov3" in backbone_name else None
    )
    model = FieldModel(
        candidate["imgsz"],
        pretrained=initial is None,
        backbone_name=backbone_name,
        pretrained_file=pretrained_file,
    ).to(device)
    if initial is not None:
        load_initial(model, initial)
    for parameter in model.backbone.parameters():
        parameter.requires_grad = False
    for block in model.backbone.blocks[-candidate["unfreeze_blocks"] :]:
        block.requires_grad_(True)
    model.backbone.norm.requires_grad_(True)
    from timm.layers import set_reentrant_ckpt

    set_reentrant_ckpt(False)
    model.backbone.set_grad_checkpointing(True)
    ddp = DDP(model, device_ids=[local_rank], broadcast_buffers=False)
    optimizer = torch.optim.AdamW(
        [
            dict(
                params=[
                    p
                    for n, p in model.named_parameters()
                    if p.requires_grad and n.startswith("backbone.")
                ],
                lr=candidate["backbone_lr"],
            ),
            dict(
                params=[
                    p
                    for n, p in model.named_parameters()
                    if p.requires_grad and (not n.startswith("backbone."))
                ],
                lr=candidate["head_lr"],
            ),
        ],
        weight_decay=0.04,
    )
    train_frame = frame[frame.split == "train"].reset_index(drop=True)
    validation = frame[frame.split == "val"].reset_index(drop=True)
    has_negative = bool((train_frame.label == "nonplant").any())
    sampler = BalancedDistributedSampler(
        balanced_weights(train_frame),
        candidate["batch_size"],
        rank,
        world,
        candidate["seed"],
        config.get("max_samples_per_epoch"),
    )
    train_loader = loader(
        train_frame,
        candidate["imgsz"],
        config["workers"],
        candidate["batch_size"],
        True,
        sampler,
    )
    accum = candidate["accum_steps"]
    updates = math.ceil(len(train_loader) / accum) * candidate["epochs"]
    train_until = float(os.environ.get("HOTBED_TRAIN_UNTIL", "inf"))
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, BudgetLR(updates, time.time(), train_until - 600.0)
    )
    scaler = torch.amp.GradScaler("cuda")
    ema = {key: value.detach().clone() for key, value in model.state_dict().items()}
    epoch_start, best_score, stale = (0, -1.0, 0)
    if config["resume"] and (output / "last.pth").exists():
        saved = torch.load(output / "last.pth", map_location=device, weights_only=True)
        if saved["fingerprint"] != fingerprint:
            raise ValueError("Resume checkpoint/config/manifest mismatch")
        model.load_state_dict(saved["state_dict"])
        optimizer.load_state_dict(saved["optimizer"])
        scheduler.load_state_dict(saved["scheduler"])
        scheduler.lr_lambdas[0].started = time.time()
        scheduler.lr_lambdas[0].deadline = train_until - 600.0
        scaler.load_state_dict(saved["scaler"])
        ema = saved["ema"]
        epoch_start, best_score, stale = (
            saved["epoch"] + 1,
            saved["best_score"],
            saved["stale"],
        )
    evaluation_reserve = 600.0
    time_stopped = False
    for epoch in range(epoch_start, candidate["epochs"]):
        if rank == 0:
            print(
                f"TRAIN: {candidate['name']} epoch={epoch + 1}, batches={len(train_loader)}",
                flush=True,
            )
        seed_all(candidate["seed"] + epoch * world + rank)
        sampler.epoch = epoch
        ddp.train()
        optimizer.zero_grad(set_to_none=True)
        total_loss, started = (0.0, time.time())
        last_progress = started
        for step, (images, labels, _) in enumerate(train_loader):
            images, labels = (images.to(device), labels.to(device))
            sync = (step + 1) % accum == 0 or step + 1 == len(train_loader)
            group_start = step // accum * accum
            actual_accum = min(accum, len(train_loader) - group_start)
            with ddp.no_sync() if not sync else contextlib.nullcontext():
                with torch.autocast("cuda", dtype=torch.float16):
                    logits, _, plant = ddp(images)
                    hard = crop_loss(logits, labels)
                    gate = F.binary_cross_entropy_with_logits(
                        plant[:, 0].float(), (labels >= 0).float()
                    )
                    loss = hard + (0.2 if has_negative else 0.0) * gate
                scaler.scale(loss / actual_accum).backward()
            if sync:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                rates = torch.tensor(
                    [g["lr"] for g in optimizer.param_groups],
                    device=device,
                    dtype=torch.float64,
                )
                dist.broadcast(rates, 0)
                for group, rate in zip(optimizer.param_groups, rates.tolist()):
                    group["lr"] = rate
                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                if scaler.get_scale() >= old_scale:
                    scheduler.step()
                    with torch.no_grad():
                        for key, value in model.state_dict().items():
                            if value.is_floating_point():
                                ema[key].lerp_(value.detach(), 0.005)
                            else:
                                ema[key].copy_(value)
            total_loss += float(loss.detach())
            if rank == 0 and (step == 0 or time.time() - last_progress >= 20):
                print(
                    candidate["name"],
                    epoch + 1,
                    step + 1,
                    "/",
                    len(train_loader),
                    "loss",
                    round(total_loss / (step + 1), 4),
                    flush=True,
                )
                last_progress = time.time()
            if sync:
                stop_time = torch.tensor(
                    int(time.time() >= train_until - evaluation_reserve), device=device
                )
                dist.all_reduce(stop_time, op=dist.ReduceOp.MAX)
                if stop_time.item():
                    time_stopped = True
                    break
        evaluation_started = time.time()
        live = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }
        model.load_state_dict(ema)
        local = infer(
            model,
            validation.iloc[rank::world],
            candidate["imgsz"],
            config["workers"],
            candidate["batch_size"],
            device,
        )
        gathered = [None] * world
        dist.all_gather_object(
            gathered, {key: local[key] for key in ["ids", "labels", "logits"]}
        )
        model.load_state_dict(live)
        if rank == 0:
            merged = {
                key: np.concatenate([part[key] for part in gathered])
                for key in ["ids", "labels", "logits"]
            }
            metrics = selection_metrics(merged, frame)
            improved = metrics["selection_score"] > best_score + 1e-05
            if improved:
                best_score, stale = (metrics["selection_score"], 0)
                atomic_save(
                    output / "best.pth",
                    dict(
                        state_dict={k: v.cpu() for k, v in ema.items()},
                        classes=CLASSES,
                        imgsz=candidate["imgsz"],
                        backbone=backbone_name,
                        epoch=epoch,
                        metrics=metrics,
                        fingerprint=fingerprint,
                        plant_gate_trained=has_negative,
                    ),
                )
            else:
                stale += 1
            atomic_save(
                output / "last.pth",
                dict(
                    state_dict=model.state_dict(),
                    ema=ema,
                    optimizer=optimizer.state_dict(),
                    scheduler=scheduler.state_dict(),
                    scaler=scaler.state_dict(),
                    epoch=epoch,
                    best_score=best_score,
                    stale=stale,
                    fingerprint=fingerprint,
                ),
            )
            record = dict(
                epoch=epoch + 1,
                loss=total_loss / (step + 1),
                batches=step + 1,
                time_stopped=time_stopped,
                seconds=time.time() - started,
                **metrics,
            )
            with open(output / "history.jsonl", "a", encoding="utf-8") as stream:
                stream.write(json.dumps(record) + "\n")
            print(
                f"Epoch {epoch + 1}: loss={record['loss']:.4f}",
                f"F1={metrics['supported_macro_f1']:.4f}",
                f"score={metrics['selection_score']:.4f}",
                flush=True,
            )
        evaluation_reserve = max(600.0, 1.5 * (time.time() - evaluation_started))
        stop = torch.tensor(
            int(
                stale >= candidate["patience"]
                or time_stopped
                or time.time() >= train_until - evaluation_reserve
            ),
            device=device,
        )
        dist.broadcast(stop, 0)
        if stop.item():
            break
    if rank == 0:
        saved = torch.load(output / "best.pth", map_location="cpu", weights_only=True)
        atomic_json(
            output / "done.json",
            dict(
                name=candidate["name"],
                imgsz=candidate["imgsz"],
                backbone=backbone_name,
                fingerprint=fingerprint,
                selection_score=saved["metrics"]["selection_score"],
                plant_gate_trained=saved["plant_gate_trained"],
                metrics=saved["metrics"],
                time_stopped=time_stopped,
            ),
        )
    dist.barrier()
    dist.destroy_process_group()


## 7. Калибровка и экспорт ONNX

In [ ]:
def scores_v3(logits, embedding, plant_logit, classes, metadata, crop_prefix=None):
    allowed = [
        i
        for i, name in enumerate(classes)
        if crop_prefix is None or name.startswith(crop_prefix + "_")
    ]
    if not allowed:
        raise ValueError(f"Unsupported crop: {crop_prefix}")
    temperature = float(metadata["temperature"])
    if not np.isfinite(temperature) or temperature <= 0:
        raise ValueError("Invalid calibration temperature")
    values = np.asarray(logits, dtype=np.float64)[allowed] / temperature
    exp = np.exp(values - values.max())
    probabilities = exp / exp.sum()
    local = int(probabilities.argmax())
    index = allowed[local]
    vector = np.asarray(embedding, dtype=np.float64)
    vector = vector / max(float(np.linalg.norm(vector)), 1e-12)
    centers = np.asarray(
        metadata["prototype_centers"][classes[index]], dtype=np.float64
    )
    centers = centers / np.maximum(
        np.linalg.norm(centers, axis=1, keepdims=True), 1e-12
    )
    distance = float(1 - np.max(centers @ vector))
    logit = float(np.asarray(plant_logit).reshape(-1)[0])
    plant_probability = float(1 / (1 + np.exp(-np.clip(logit, -80, 80))))
    return (index, float(probabilities[local]), distance, plant_probability)


In [ ]:
def decide_v3(logits, embedding, plant_logit, classes, metadata, crop_prefix=None):
    index, confidence, distance, plant_probability = scores_v3(
        logits, embedding, plant_logit, classes, metadata, crop_prefix
    )
    name = classes[index]
    threshold = float(metadata["prototype_distance_thresholds"][name])
    confidence_threshold = float(metadata["abstain_threshold"])
    if name.endswith("_healthy"):
        confidence_threshold = max(
            confidence_threshold, float(metadata["healthy_threshold"])
        )
    if metadata.get("plant_gate_enabled") and plant_probability < float(
        metadata["plant_threshold"]
    ):
        reason = "out_of_distribution"
    elif confidence < confidence_threshold:
        reason = "low_confidence"
    elif distance > threshold:
        reason = "out_of_distribution"
    else:
        reason = None
    return dict(
        class_index=index,
        class_name=name,
        confidence=confidence,
        prototype_distance=distance,
        accepted=reason is None,
        rejection_reason=reason,
        distance_threshold=threshold,
    )


In [ ]:
@dataclass(frozen=True)
class OnnxPrediction:
    class_index: int
    class_name: str
    confidence: float
    prototype_distance: float | None
    accepted: bool
    rejection_reason: str | None
    distance_threshold: float | None = None


In [ ]:
class InvalidImageQuality(ValueError):
    """Raised when an image contains too little visual information for diagnosis."""

    def __init__(self, reason: str):
        super().__init__(reason)
        self.reason = reason


In [ ]:
def stable_softmax(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)
    exponent = np.exp(values - np.max(values))
    return exponent / max(float(exponent.sum()), 1e-12)


In [ ]:
class PlantDiseaseOnnxClassifier:

    def __init__(
        self,
        model_path: str,
        metadata_path: str,
        providers: Sequence[str] | None = None,
    ):
        try:
            import onnxruntime as ort
        except ImportError as exc:
            raise RuntimeError(
                'onnxruntime is not installed'
            ) from exc
        metadata = json.loads(Path(metadata_path).read_text(encoding="utf-8"))
        self.metadata = metadata
        self.decision_policy_version = int(metadata.get("decision_policy_version", 2))
        if self.decision_policy_version not in (2, 3):
            raise ValueError(
                'Unsupported decision policy'
            )
        self.classes = [str(value) for value in metadata["classes"]]
        self.model_name = str(metadata.get("model", "unknown"))
        self.img_size = int(metadata.get("imgsz", 392))
        self.threshold = float(metadata.get("abstain_threshold", 0.8))
        normalization = metadata.get("normalization", {})
        self.mean = np.asarray(
            normalization.get("mean", [0.485, 0.456, 0.406]), dtype=np.float32
        ).reshape(1, 1, 3)
        self.std = np.asarray(
            normalization.get("std", [0.229, 0.224, 0.225]), dtype=np.float32
        ).reshape(1, 1, 3)
        raw_prototypes = metadata.get("prototypes")
        self.prototypes = (
            np.asarray(raw_prototypes, dtype=np.float32)
            if raw_prototypes is not None
            else None
        )
        self.distance_thresholds = {
            str(key): float(value)
            for key, value in metadata.get("prototype_distance_thresholds", {}).items()
        }
        self.field_distance_multiplier = float(
            os.environ.get("MODEL_FIELD_DISTANCE_MULTIPLIER", "2.5")
        )
        self.field_distance_floor = float(
            os.environ.get("MODEL_FIELD_DISTANCE_FLOOR", "0.04")
        )
        self.field_min_confidence = float(
            os.environ.get("MODEL_FIELD_MIN_CONFIDENCE", "0.70")
        )
        self.field_healthy_min_confidence = float(
            os.environ.get("MODEL_FIELD_HEALTHY_MIN_CONFIDENCE", "0.90")
        )
        if providers is None:
            available = set(ort.get_available_providers())
            providers = [
                name
                for name in ("CUDAExecutionProvider", "CPUExecutionProvider")
                if name in available
            ]
        self.session = ort.InferenceSession(str(model_path), providers=list(providers))
        self.input_name = self.session.get_inputs()[0].name

    def preprocess(self, image_path: str) -> np.ndarray:
        with Image.open(image_path) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")
            image = image.resize(
                (self.img_size, self.img_size), Image.Resampling.BICUBIC
            )
            array = np.asarray(image, dtype=np.float32) / 255.0
        luminance = (
            0.2126 * array[:, :, 0] + 0.7152 * array[:, :, 1] + 0.0722 * array[:, :, 2]
        )
        mean_luminance = float(luminance.mean())
        p05, median_luminance, p95 = (
            float(value) for value in np.percentile(luminance, (5, 50, 95))
        )
        contrast = p95 - p05
        horizontal_detail = float(np.mean(np.abs(np.diff(luminance, axis=1))))
        vertical_detail = float(np.mean(np.abs(np.diff(luminance, axis=0))))
        detail = (horizontal_detail + vertical_detail) / 2.0
        strong_edge_fraction = (
            float(np.mean(np.abs(np.diff(luminance, axis=1)) > 0.035))
            + float(np.mean(np.abs(np.diff(luminance, axis=0)) > 0.035))
        ) / 2.0
        red, green, blue = (array[:, :, 0], array[:, :, 1], array[:, :, 2])
        warm_red_fraction = float(
            np.mean((red > 0.15) & (red - green > 0.065) & (red - blue > 0.055))
        )
        plant_green_fraction = float(
            np.mean((green > 0.12) & (green > red * 1.04) & (green > blue * 1.07))
        )
        if (
            mean_luminance < 0.055
            or p95 < 0.1
            or (median_luminance < 0.08 and p95 < 0.22)
            or (p95 < 0.18 and contrast < 0.1 and (detail < 0.015))
        ):
            raise InvalidImageQuality("image_too_dark")
        if mean_luminance > 0.985:
            raise InvalidImageQuality("image_too_bright")
        if (
            warm_red_fraction > 0.62
            and plant_green_fraction < 0.015
            and (strong_edge_fraction < 0.12 or detail < 0.035)
        ):
            raise InvalidImageQuality("camera_obscured")
        if contrast < 0.035 or float(luminance.std()) < 0.012:
            raise InvalidImageQuality("blank_image")
        if strong_edge_fraction < 0.006 or (
            detail < 0.008 and strong_edge_fraction < 0.02
        ):
            raise InvalidImageQuality("image_too_blurry")
        array = (array - self.mean) / self.std
        return np.transpose(array, (2, 0, 1))[None].astype(np.float32, copy=False)

    def predict(
        self, image_path: str, crop_prefix: str | None = None
    ) -> OnnxPrediction:
        try:
            model_input = self.preprocess(image_path)
        except InvalidImageQuality as exc:
            return OnnxPrediction(-1, "unknown", 0.0, None, False, exc.reason, None)
        outputs = self.session.run(None, {self.input_name: model_input})
        if getattr(self, "decision_policy_version", 2) == 3:
            if len(outputs) != 3:
                raise ValueError(
                    'Missing ONNX outputs'
                )
            return OnnxPrediction(
                **decide_v3(
                    outputs[0][0],
                    outputs[1][0],
                    outputs[2][0],
                    self.classes,
                    self.metadata,
                    crop_prefix,
                )
            )
        logits = np.asarray(outputs[0][0], dtype=np.float32)
        embedding = (
            np.asarray(outputs[1][0], dtype=np.float32) if len(outputs) > 1 else None
        )
        allowed = [
            index
            for index, class_name in enumerate(self.classes)
            if crop_prefix is None or class_name.startswith(crop_prefix + "_")
        ]
        if not allowed:
            allowed = list(range(len(self.classes)))
        probabilities = stable_softmax(logits[allowed])
        local_index = int(np.argmax(probabilities))
        class_index = int(allowed[local_index])
        class_name = self.classes[class_index]
        confidence = float(probabilities[local_index])
        distance = None
        effective_distance_threshold = None
        distance_ok = True
        if (
            embedding is not None
            and self.prototypes is not None
            and (class_index < len(self.prototypes))
        ):
            norm = max(float(np.linalg.norm(embedding)), 1e-12)
            embedding = embedding / norm
            distance = float(1.0 - np.dot(embedding, self.prototypes[class_index]))
            strict_distance_threshold = self.distance_thresholds.get(class_name, 0.45)
            field_distance_multiplier = getattr(self, "field_distance_multiplier", 2.5)
            field_distance_floor = getattr(self, "field_distance_floor", 0.04)
            field_min_confidence = getattr(self, "field_min_confidence", 0.7)
            field_healthy_min_confidence = getattr(
                self, "field_healthy_min_confidence", 0.9
            )
            relaxed_min_confidence = (
                field_healthy_min_confidence
                if class_name.endswith("_healthy")
                else field_min_confidence
            )
            effective_distance_threshold = max(
                strict_distance_threshold,
                min(
                    0.45,
                    max(
                        strict_distance_threshold * field_distance_multiplier,
                        field_distance_floor,
                    ),
                ),
            )
            distance_ok = bool(
                distance <= strict_distance_threshold
                or (
                    confidence >= max(self.threshold, relaxed_min_confidence)
                    and distance <= effective_distance_threshold
                )
            )
        confidence_ok = confidence >= self.threshold
        accepted = bool(confidence_ok and distance_ok)
        if not confidence_ok:
            reason = "low_confidence"
        elif not distance_ok:
            reason = "out_of_distribution"
        else:
            reason = None
        return OnnxPrediction(
            class_index,
            class_name,
            confidence,
            distance,
            accepted,
            reason,
            effective_distance_threshold,
        )


In [ ]:
def quality_flags(frame, size):
    classifier = PlantDiseaseOnnxClassifier.__new__(PlantDiseaseOnnxClassifier)
    classifier.img_size = size
    classifier.mean = np.asarray(MEAN, dtype=np.float32).reshape(1, 1, 3)
    classifier.std = np.asarray(STD, dtype=np.float32).reshape(1, 1, 3)
    accepted, reasons = ({}, {})
    for row in frame.itertuples():
        try:
            classifier.preprocess(row.path)
            accepted[row.row_id], reasons[row.row_id] = (True, None)
        except InvalidImageQuality as error:
            accepted[row.row_id], reasons[row.row_id] = (False, error.reason)
    return (accepted, reasons)


In [ ]:
def build_centers(data, count):
    from sklearn.cluster import KMeans

    centers = {}
    for i, name in enumerate(CLASSES):
        vectors = data["embeddings"][data["labels"] == i]
        if not len(vectors):
            raise ValueError(f"Missing train prototypes: {name}")
        number = min(int(count), max(1, len(vectors) // 20))
        if number == 1:
            values = vectors.mean(0, keepdims=True)
        else:
            values = (
                KMeans(number, n_init=10, random_state=42).fit(vectors).cluster_centers_
            )
        values /= np.maximum(np.linalg.norm(values, axis=1, keepdims=True), 1e-12)
        centers[name] = values.tolist()
    return centers


In [ ]:
def expanded_scores(data, metadata, quality):
    """Positive samples use their known crop; negatives must be rejected for EVERY crop."""
    result = []
    for j, label in enumerate(data["labels"]):
        crops = [CROPS[int(CLASS_CROP[label])]] if label >= 0 else CROPS
        for crop in crops:
            index, confidence, distance, plant = scores_v3(
                data["logits"][j],
                data["embeddings"][j],
                data["plants"][j],
                CLASSES,
                metadata,
                crop,
            )
            result.append(
                (
                    int(data["ids"][j]),
                    int(label),
                    index,
                    confidence,
                    distance,
                    plant,
                    bool(quality[int(data["ids"][j])]),
                    crop,
                )
            )
    return pd.DataFrame(
        result,
        columns=[
            "row_id",
            "label",
            "pred",
            "confidence",
            "distance",
            "plant_probability",
            "quality_ok",
            "crop",
        ],
    )


In [ ]:
def policy_keep(scores, metadata):
    healthy = np.asarray([CLASSES[i].endswith("_healthy") for i in scores.pred])
    thresholds = np.asarray(
        [metadata["prototype_distance_thresholds"][CLASSES[i]] for i in scores.pred]
    )
    minimum = np.where(
        healthy, metadata["healthy_threshold"], metadata["abstain_threshold"]
    )
    gate = (
        scores.plant_probability.to_numpy() >= metadata["plant_threshold"]
        if metadata["plant_gate_enabled"]
        else np.ones(len(scores), dtype=bool)
    )
    return (
        (scores.confidence.to_numpy() >= minimum)
        & (scores.distance.to_numpy() <= thresholds)
        & gate
        & scores.quality_ok.to_numpy()
    )


In [ ]:
def acceptance_metrics(scores, keep):
    plants = scores.label.to_numpy() >= 0
    correct = scores.label.to_numpy() == scores.pred.to_numpy()
    accepted = keep & plants
    neg = scores.loc[~plants, ["row_id"]].copy()
    neg["accepted"] = keep[~plants]
    negative_any = (
        neg.groupby("row_id").accepted.max() if len(neg) else pd.Series(dtype=bool)
    )
    diseased = np.asarray(
        [
            label >= 0 and (not CLASSES[label].endswith("_healthy"))
            for label in scores.label
        ]
    )
    healthy_pred = np.asarray([CLASSES[i].endswith("_healthy") for i in scores.pred])
    return dict(
        coverage=float(accepted.sum() / max(1, plants.sum())),
        accepted_accuracy=float(correct[accepted].mean()) if accepted.any() else 0.0,
        correct_and_accepted_fraction=float(
            (accepted & correct).sum() / max(1, plants.sum())
        ),
        false_healthy=float(
            (keep & diseased & healthy_pred).sum() / max(1, diseased.sum())
        ),
        negative_samples=len(negative_any),
        negative_acceptance=float(negative_any.mean()) if len(negative_any) else None,
        quality_rejection_fraction=(
            float(1 - scores.quality_ok.to_numpy()[plants].mean())
            if plants.any()
            else None
        ),
        accepted=int(accepted.sum()),
        correct_accepted=int((accepted & correct).sum()),
    )


In [ ]:
def calibrate(data, prototype_data, config, plant_gate, quality):
    from scipy.optimize import minimize_scalar
    from scipy.special import logsumexp

    labels = data["labels"]
    valid = labels >= 0
    logits = data["logits"][valid].astype(np.float64)
    targets = labels[valid]
    allowed = CLASS_CROP.numpy()[None, :] == CLASS_CROP.numpy()[targets, None]

    def nll(log_temperature):
        values = np.where(allowed, logits / np.exp(log_temperature), -1000000000.0)
        return float(
            np.mean(
                logsumexp(values, axis=1) - values[np.arange(len(targets)), targets]
            )
        )

    optimum = minimize_scalar(
        nll, bounds=(math.log(0.25), math.log(4)), method="bounded"
    )
    metadata = dict(
        temperature=float(np.exp(optimum.x)),
        prototype_centers=build_centers(prototype_data, config["prototype_centers"]),
    )
    scores = expanded_scores(data, metadata, quality)
    good = scores[(scores.label >= 0) & (scores.label == scores.pred)]
    if good.empty:
        raise ValueError("No correct calibration predictions")
    pooled = max(0.001, float(good.distance.quantile(0.985)))
    thresholds = {}
    for i, name in enumerate(CLASSES):
        distances = good.loc[good.label == i, "distance"]
        weight = len(distances) / (len(distances) + 30)
        local = float(distances.quantile(0.985)) if len(distances) else pooled
        thresholds[name] = max(0.001, weight * local + (1 - weight) * pooled)
    plant_probabilities = scores.loc[scores.label >= 0, "plant_probability"].to_numpy()
    plant_thresholds = (
        sorted(
            set(
                [0.0]
                + [
                    float(np.quantile(plant_probabilities, q))
                    for q in [0.005, 0.01, 0.02]
                ]
            )
        )
        if plant_gate
        else [0.0]
    )
    trials = []
    for multiplier in [1.0, 1.25, 1.5]:
        for threshold in np.arange(0.35, 0.951, 0.025):
            for healthy_offset in [0.0, 0.1, 0.2]:
                for plant_threshold in plant_thresholds:
                    policy = dict(
                        metadata,
                        plant_gate_enabled=plant_gate,
                        plant_threshold=plant_threshold,
                        abstain_threshold=float(threshold),
                        healthy_threshold=min(0.99, float(threshold + healthy_offset)),
                        prototype_distance_thresholds={
                            k: min(0.8, v * multiplier) for k, v in thresholds.items()
                        },
                    )
                    keep = policy_keep(scores, policy)
                    metrics = acceptance_metrics(scores, keep)
                    plant_recall = (
                        float((plant_probabilities >= plant_threshold).mean())
                        if plant_gate
                        else 1.0
                    )
                    feasible = (
                        metrics["accepted_accuracy"]
                        >= config["calibration_target_accuracy"]
                        and metrics["coverage"] >= config["minimum_coverage"]
                        and (
                            metrics["false_healthy"] <= config["maximum_false_healthy"]
                        )
                        and (plant_recall >= config["calibration_min_plant_recall"])
                        and (
                            metrics["negative_acceptance"] is None
                            or metrics["negative_acceptance"]
                            <= config["calibration_max_negative_acceptance"]
                        )
                    )
                    trials.append(
                        (
                            feasible,
                            (
                                metrics["coverage"]
                                if feasible
                                else metrics["correct_and_accepted_fraction"]
                            ),
                            metrics["accepted_accuracy"],
                            policy,
                            metrics,
                        )
                    )
    best = max(trials, key=lambda item: item[:3])
    result = best[3]
    result["calibration_constraints_passed"] = bool(best[0])
    result["calibration_metrics"] = best[4]
    result["calibration_temperature_nll_before"] = nll(0.0)
    result["calibration_temperature_nll_after"] = nll(math.log(result["temperature"]))
    return result


In [ ]:
def summarize(data, metadata, frame, quality):
    predicted = conditional_predictions(data["logits"], data["labels"])
    metrics = classification_metrics(data["labels"], predicted)
    scores = expanded_scores(data, metadata, quality)
    keep = policy_keep(scores, metadata)
    metrics.update(acceptance_metrics(scores, keep))
    scores["accepted"] = keep
    metrics["by_crop"] = {}
    for crop in CROPS:
        selected = scores.crop.to_numpy() == crop
        if ((scores.label.to_numpy() >= 0) & selected).any():
            metrics["by_crop"][crop] = acceptance_metrics(
                scores[selected], keep[selected]
            )
    return (metrics, scores)


In [ ]:
def verify_export(
    model, metadata, frame, artifact, config, device, checker_python=None
):
    import onnx
    import onnxruntime as ort

    model.backbone.set_grad_checkpointing(False)
    for module in model.modules():
        if hasattr(module, "fused_attn"):
            module.fused_attn = False
    model = model.to("cpu").eval()
    size = metadata["imgsz"]
    dummy = torch.zeros(1, 3, size, size)
    path = artifact / "distilled_student.onnx"
    torch.onnx.export(
        model,
        dummy,
        str(path),
        input_names=["images"],
        output_names=["logits", "embedding", "plant_logit"],
        opset_version=17,
        do_constant_folding=True,
        dynamo=False,
    )
    if checker_python:
        import subprocess

        subprocess.run(
            [
                str(checker_python),
                "-c",
                'import sys; sys.path.insert(0, sys.argv[1]); import onnx; onnx.checker.check_model(sys.argv[2]); print("Isolated ONNX checker OK")',
                str(Path(onnx.__file__).resolve().parents[1]),
                str(path),
            ],
            check=True,
        )
    else:
        onnx.checker.check_model(str(path))
    options = ort.SessionOptions()
    options.intra_op_num_threads = min(4, os.cpu_count() or 1)
    session = ort.InferenceSession(
        str(path), sess_options=options, providers=["CPUExecutionProvider"]
    )
    sample = frame.groupby("label", group_keys=False).head(1).head(40)
    maximum_errors = [0.0, 0.0, 0.0]
    times = []
    with torch.inference_mode():
        for image, _, _ in loader(sample, size, 0, 1):
            reference = [value.numpy() for value in model(image)]
            start = time.perf_counter()
            output = session.run(None, {"images": image.numpy()})
            times.append(time.perf_counter() - start)
            for i, (actual, expected) in enumerate(zip(output, reference)):
                if actual.shape != expected.shape or not np.all(np.isfinite(actual)):
                    raise ValueError("Invalid ONNX output contract")
                np.testing.assert_allclose(actual, expected, rtol=0.002, atol=0.002)
                maximum_errors[i] = max(
                    maximum_errors[i], float(np.max(np.abs(actual - expected)))
                )
            assert [tuple(x.shape) for x in output] == [(1, 39), (1, 384), (1, 1)]
    measured = times[1:] or times
    report = dict(
        max_abs_error=maximum_errors,
        samples=len(times),
        provider="CPUExecutionProvider",
        structural_check_passed=True,
        checker_python=str(checker_python) if checker_python else "current interpreter",
        threads=options.intra_op_num_threads,
        p50_seconds=float(np.median(measured)),
        p95_seconds=float(np.quantile(measured, 0.95)),
        scope="ONNX model time on host CPU; excludes HTTP and recommendations",
    )
    atomic_json(artifact / "onnx_verification.json", report)
    return (session, report)


In [ ]:
def evaluate_serving(session, metadata, frame):
    """Fresh ONNX predictions + the exact quality/policy path on final holdouts."""
    classifier = PlantDiseaseOnnxClassifier.__new__(PlantDiseaseOnnxClassifier)
    classifier.img_size = metadata["imgsz"]
    classifier.mean = np.asarray(MEAN, dtype=np.float32).reshape(1, 1, 3)
    classifier.std = np.asarray(STD, dtype=np.float32).reshape(1, 1, 3)
    classifier.session = session
    classifier.input_name = session.get_inputs()[0].name
    classifier.metadata = metadata
    classifier.classes = CLASSES
    classifier.decision_policy_version = 3
    records = []
    for row in frame.itertuples():
        label = CLASSES.index(row.label) if row.label in CLASSES else -1
        for crop in [row.label.split("_")[0]] if label >= 0 else CROPS:
            prediction = classifier.predict(row.path, crop)
            records.append(
                dict(
                    row_id=row.row_id,
                    label=label,
                    pred=prediction.class_index,
                    confidence=prediction.confidence,
                    distance=prediction.prototype_distance,
                    crop=crop,
                    accepted=prediction.accepted,
                    quality_ok=prediction.rejection_reason
                    not in {
                        "image_too_dark",
                        "image_too_bright",
                        "blank_image",
                        "image_too_blurry",
                        "camera_obscured",
                    },
                    rejection_reason=prediction.rejection_reason,
                    source=row.source,
                )
            )
    scores = pd.DataFrame(records)
    scores["pred"] = scores.pred.where(scores.pred >= 0, 0)
    return (acceptance_metrics(scores, scores.accepted.to_numpy()), scores)


In [ ]:
def select_deployable(config, work, frame, completed):
    """Select by validation score and verify ONNX export."""
    import gc

    reports = []
    for candidate in sorted(
        completed, key=lambda item: item["selection_score"], reverse=True
    ):
        directory = work / candidate["name"] / "deployment_preflight"
        directory.mkdir(exist_ok=True)
        checkpoint_path = work / candidate["name"] / "best.pth"
        try:
            model = FieldModel(candidate["imgsz"], backbone_name=candidate["backbone"])
            load_initial(model, checkpoint_path)
            session, report = verify_export(
                model,
                {"imgsz": candidate["imgsz"]},
                frame[frame.split == "val"],
                directory,
                config,
                torch.device("cpu"),
            )
            reports.append(
                dict(
                    name=candidate["name"],
                    export_passed=True,
                    p95_seconds=report["p95_seconds"],
                )
            )
            del session, model
            gc.collect()
            return (candidate, directory, report, reports)
        except OSError:
            raise
        except Exception as error:
            reports.append(
                dict(name=candidate["name"], export_passed=False, error=str(error))
            )
            print('Model check failed:', candidate['name'], str(error), flush=True)
            if "model" in locals():
                del model
            gc.collect()
    atomic_json(work / "deployment_failures.json", reports)
    raise RuntimeError(
        'ONNX export failed'
    )


In [ ]:
def export_notebook_runtime(artifact):
    (artifact / "onnx_classifier.py").write_text(RUNTIME_SOURCE, encoding="utf-8")
    (artifact / "field_v3_policy.py").write_text(POLICY_SOURCE, encoding="utf-8")
    (artifact / "final_train.ipynb").write_text(NOTEBOOK_SNAPSHOT, encoding="utf-8")


In [ ]:
def finalize(config):
    work = Path(config["work_dir"])
    frame = pd.read_csv(work / "split_manifest.csv").fillna("")
    completed = [
        json.loads((work / c["name"] / "done.json").read_text())
        for c in active_candidates(config)
        if (work / c["name"] / "done.json").exists()
    ]
    if not completed:
        raise RuntimeError(
            'Trained checkpoint not found'
        )
    selection_path = work / "selection.json"
    frozen = json.loads(selection_path.read_text()) if selection_path.exists() else None
    if frozen and (work / "release_candidate" / "finalization_id.json").exists():
        winner = frozen["winner"]
        preflight_dir = work / winner["name"] / "deployment_preflight"
        verification = json.loads(
            (preflight_dir / "onnx_verification.json").read_text()
        )
        deployment_reports = frozen["deployment_preflight"]
    else:
        winner, preflight_dir, verification, deployment_reports = select_deployable(
            config, work, frame, completed
        )
    atomic_json(
        selection_path,
        dict(
            winner=winner,
            candidates=completed,
            selected_on="val score and valid ONNX export",
            deployment_preflight=deployment_reports,
        ),
    )
    checkpoint_path = work / winner["name"] / "best.pth"
    finalization_id = hashlib.sha256(
        (
            sha_file(checkpoint_path)
            + sha_file(work / "split_manifest.csv")
            + json.dumps(config, sort_keys=True)
            + NOTEBOOK_DIGEST
        ).encode()
    ).hexdigest()
    artifact = work / "release_candidate"
    artifact.mkdir(exist_ok=True)
    state_path = artifact / "finalization_id.json"
    if (
        state_path.exists()
        and json.loads(state_path.read_text())["id"] != finalization_id
    ):
        raise ValueError(
            'Finalization inputs changed'
        )
    atomic_json(state_path, dict(id=finalization_id))
    if (work / "hotbed_field_v3.zip").exists() and (artifact / "SHA256.json").exists():
        print("Finalized archive already exists:", work / "hotbed_field_v3.zip")
        return
    saved = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    model = FieldModel(winner["imgsz"], backbone_name=winner["backbone"])
    load_initial(model, checkpoint_path)
    device = torch.device("cuda")
    model.to(device).eval()
    training = frame[(frame.split == "train") & (frame.label != "nonplant")]
    prototype_frame = (
        training.groupby(["label", "source"], group_keys=False)
        .sample(frac=1.0, random_state=config["seed"])
        .groupby(["label", "source"], group_keys=False)
        .head(config["prototype_samples_per_class_source"])
    )
    proto = infer(model, prototype_frame, winner["imgsz"], config["workers"], 2, device)
    cal_frame = frame[frame.split == "calibration"]
    cal = infer(model, cal_frame, winner["imgsz"], config["workers"], 2, device)
    quality, _ = quality_flags(cal_frame, winner["imgsz"])
    metadata = calibrate(cal, proto, config, winner["plant_gate_trained"], quality)
    metadata.update(
        format_version=3,
        decision_policy_version=3,
        model="field_v3_global_local",
        backbone=winner["backbone"],
        imgsz=winner["imgsz"],
        classes=CLASSES,
        normalization=dict(mean=MEAN, std=STD),
        plant_gate_trained=winner["plant_gate_trained"],
        selection=winner,
        metadata_scope="Calibrated using public held-out groups; own-phone data unavailable",
        deployment_requires=["onnx_classifier.py", "field_v3_policy.py"],
    )
    np.savez_compressed(artifact / "calibration_outputs.npz", **cal)
    atomic_json(artifact / "model_metadata.json", metadata)
    shutil.copy2(
        preflight_dir / "distilled_student.onnx", artifact / "distilled_student.onnx"
    )
    shutil.copy2(
        preflight_dir / "onnx_verification.json", artifact / "onnx_verification.json"
    )
    import onnxruntime as ort

    if "CUDAExecutionProvider" in ort.get_available_providers():
        session = ort.InferenceSession(
            str(artifact / "distilled_student.onnx"),
            providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
        )
        if "CUDAExecutionProvider" not in session.get_providers():
            raise RuntimeError(
                'ONNX CUDA session failed'
            )
    else:
        session = ort.InferenceSession(
            str(artifact / "distilled_student.onnx"), providers=["CPUExecutionProvider"]
        )
        print('Evaluation on CPU', flush=True)
    model.to(device)
    evaluations, tables = ({}, [])
    for split in ["field_test", "domain_test"]:
        part = frame[frame.split == split]
        outputs = infer(model, part, winner["imgsz"], config["workers"], 2, device)
        np.savez_compressed(artifact / f"{split}_outputs.npz", **outputs)
        metrics = classification_metrics(
            outputs["labels"],
            conditional_predictions(outputs["logits"], outputs["labels"]),
        )
        serving, table = evaluate_serving(session, metadata, part)
        metrics.update(serving)
        metrics["classification_scope"] = "All plant images, before rejection"
        metrics["acceptance_scope"] = "Fresh ONNX plus deployed quality and v3 policy"
        evaluations[split] = metrics
        table["split"] = split
        tables.append(table)
    runtime = PlantDiseaseOnnxClassifier(
        str(artifact / "distilled_student.onnx"),
        str(artifact / "model_metadata.json"),
        ["CPUExecutionProvider"],
    )
    example = frame[frame.split == "val"].iloc[0]
    runtime.predict(
        example.path,
        example.label.split("_")[0] if example.label in CLASSES else "tomato",
    )
    negatives = {
        split: int(((frame.split == split) & (frame.label == "nonplant")).sum())
        for split in ["train", "calibration", "field_test"]
    }
    metadata.update(
        evaluation_metrics=evaluations,
        negative_split_counts=negatives,
        onnx_verification=verification,
        manifest_sha256=sha_file(work / "split_manifest.csv"),
        checkpoint_sha256=sha_file(checkpoint_path),
        runtime_policy_overrides="Decision thresholds are taken from calibrated model metadata.",
    )
    atomic_json(artifact / "model_metadata.json", metadata)
    pd.concat(tables, ignore_index=True).to_csv(
        artifact / "predictions.csv", index=False
    )
    (artifact / "classes.txt").write_text("\n".join(CLASSES) + "\n", encoding="utf-8")
    shutil.copy2(checkpoint_path, artifact / "best.pth")
    for name in [
        "split_manifest.csv",
        "split_counts.csv",
        "rejected_images.csv",
        "selection.json",
    ]:
        shutil.copy2(work / name, artifact / name)
    if (work / "budget_report.json").exists():
        shutil.copy2(work / "budget_report.json", artifact / "budget_report.json")
    if (work / "negative_selection.json").exists():
        shutil.copy2(
            work / "negative_selection.json", artifact / "negative_selection.json"
        )
    export_notebook_runtime(artifact)
    atomic_json(artifact / "training_config.json", config)
    checksums = {p.name: sha_file(p) for p in artifact.iterdir() if p.is_file()}
    atomic_json(artifact / "SHA256.json", checksums)
    archive = work / "hotbed_field_v3.zip"
    with zipfile.ZipFile(
        archive, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1
    ) as bundle:
        for path in artifact.iterdir():
            if path.is_file():
                bundle.write(path, path.name)
    with zipfile.ZipFile(archive) as bundle:
        if bundle.testzip() is not None:
            raise ValueError("Archive CRC verification failed")
    print("Selected model:", winner["name"], flush=True)
    print("Archive:", archive, flush=True)


## 8. Запуск процессов

In [ ]:
def notebook_worker(function, arguments, options):
    if os.name == "posix":
        os.setsid()
    os.environ.update(
        RANK=str(options["rank"]),
        LOCAL_RANK=str(options["rank"]),
        WORLD_SIZE=str(options["processes"]),
        MASTER_ADDR="127.0.0.1",
        MASTER_PORT=str(options["port"]),
        PYTHONUNBUFFERED="1",
        OMP_NUM_THREADS="2",
        TOKENIZERS_PARALLELISM="false",
    )
    if options["train_until"] is not None:
        os.environ["HOTBED_TRAIN_UNTIL"] = str(options["train_until"])
    with open(options["log_path"], "w", encoding="utf-8", buffering=1) as stream:
        with contextlib.redirect_stdout(stream), contextlib.redirect_stderr(stream):
            try:
                function(*arguments)
            except BaseException:
                traceback.print_exc()
                raise
            finally:
                if dist.is_initialized():
                    dist.destroy_process_group()


In [ ]:
def run_in_workers(
    function, arguments, *, work, label, timeout, processes=1, train_until=None
):
    if timeout <= 0:
        raise TimeoutError("Time budget exhausted")
    log_dir = Path(work) / "logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    with socket.socket() as listener:
        listener.bind(("127.0.0.1", 0))
        port = listener.getsockname()[1]
    payload = cloudpickle.dumps((notebook_worker, function, arguments))
    bootstrap = (
        "import cloudpickle\n"
        "worker, function, arguments = cloudpickle.loads(payload)\n"
        "worker(function, arguments, options)"
    )
    context = multiprocessing.get_context("spawn")
    children, readers = [], []
    started = last_heartbeat = time.monotonic()

    def drain():
        for rank, reader in enumerate(readers):
            chunk = reader.read()
            if chunk:
                print(
                    f"[rank {rank}] {chunk}",
                    end="" if chunk.endswith("\n") else "\n",
                    flush=True,
                )

    try:
        for rank in range(processes):
            log_path = log_dir / f"{label}_rank{rank}.log"
            log_path.write_text("", encoding="utf-8")
            readers.append(log_path.open(encoding="utf-8", errors="replace"))
            options = dict(
                rank=rank,
                processes=processes,
                port=port,
                train_until=train_until,
                log_path=str(log_path),
            )
            child = context.Process(
                target=exec, args=(bootstrap, {"payload": payload, "options": options})
            )
            child.start()
            children.append(child)
        while any(child.is_alive() for child in children):
            drain()
            failed = [child for child in children if child.exitcode not in (None, 0)]
            if failed:
                raise RuntimeError(
                    f"{label}: exit code {failed[0].exitcode}"
                )
            elapsed = time.monotonic() - started
            if elapsed >= timeout:
                raise TimeoutError(f"{label}: timeout ({timeout:.0f}s)")
            if time.monotonic() - last_heartbeat >= 20:
                print(f"{label}: running for {elapsed:.0f}s", flush=True)
                last_heartbeat = time.monotonic()
            time.sleep(0.2)
        for child in children:
            child.join()
        drain()
        if any(child.exitcode != 0 for child in children):
            raise RuntimeError(
                f"{label}: worker failed"
            )
    finally:
        for child in children:
            if child.is_alive():
                if os.name == "posix":
                    try:
                        os.killpg(child.pid, signal.SIGTERM)
                    except ProcessLookupError:
                        child.terminate()
                else:
                    child.terminate()
        for child in children:
            child.join(timeout=3)
            if child.is_alive():
                if os.name == "posix":
                    try:
                        os.killpg(child.pid, signal.SIGKILL)
                    except ProcessLookupError:
                        child.kill()
                else:
                    child.kill()
                child.join(timeout=3)
        drain()
        for reader in readers:
            reader.close()


In [ ]:
def run_training(config, session_started):
    work = Path(config["work_dir"])
    end, train_end, _ = deadlines(config, session_started, time.time())
    records = []
    for candidate in config["candidates"]:
        if candidate.get("enabled_by") and (
            not config.get(candidate["enabled_by"], False)
        ):
            continue
        if (work / candidate["name"] / "done.json").exists():
            saved = json.loads((work / candidate["name"] / "done.json").read_text())
            fingerprint = run_fingerprint(config, candidate)
            if saved["fingerprint"] != fingerprint:
                raise ValueError(
                    'Checkpoint/config mismatch'
                )
            records.append(dict(name=candidate["name"], status="already_completed"))
            continue
        remaining = train_end - time.time()
        if remaining < 1800:
            records.append(dict(name=candidate["name"], status="skipped_time_budget"))
            continue
        until = min(
            train_end, time.time() + remaining * candidate.get("budget_fraction", 1.0)
        )
        print(
            "Training:",
            candidate["name"],
            "budget minutes:",
            round((until - time.time()) / 60),
            flush=True,
        )
        try:
            run_in_workers(
                train,
                (config, candidate),
                work=work,
                label=candidate["name"],
                processes=2,
                train_until=until,
                timeout=max(1.0, train_end - time.time()),
            )
        except (TimeoutError, RuntimeError) as error:
            checkpoint = work / candidate["name"] / "best.pth"
            if checkpoint.exists():
                import torch

                saved = torch.load(checkpoint, map_location="cpu", weights_only=True)
                if saved["fingerprint"] != run_fingerprint(config, candidate):
                    raise ValueError(
                        'Checkpoint experiment mismatch'
                    ) from error
                recovered = dict(
                    name=candidate["name"],
                    imgsz=saved["imgsz"],
                    backbone=saved["backbone"],
                    fingerprint=saved["fingerprint"],
                    metrics=saved["metrics"],
                    selection_score=saved["metrics"]["selection_score"],
                    plant_gate_trained=saved["plant_gate_trained"],
                    recovered_after_failure=str(error),
                )
                (checkpoint.parent / "done.json").write_text(json.dumps(recovered))
                del saved
            elif not any(
                (
                    (work / c["name"] / "done.json").exists()
                    for c in config["candidates"]
                )
            ):
                raise
            records.append(
                dict(
                    name=candidate["name"],
                    status="interrupted_using_last_validated_result",
                    error=str(error),
                )
            )
            print('Training interrupted. Saving best checkpoint.\n', error, flush=True)
            break
        records.append(
            dict(name=candidate["name"], status="completed", finished=time.time())
        )
    (work / "budget_report.json").write_text(
        json.dumps(
            dict(
                session_started=session_started,
                planned_finish=end,
                training_deadline=train_end,
                finalization_started=time.time(),
                candidates=records,
            ),
            indent=2,
        )
    )


In [ ]:
def prepare(config):
    if config["backbone"] not in BACKBONES or config["embedding_dim"] != 384:
        raise ValueError("Unsupported backbone or embedding dimension")
    for candidate in active_candidates(config):
        name = candidate.get("backbone", config["backbone"])
        patch_size = 16 if "dinov3" in name else 14
        if name not in BACKBONES or candidate["imgsz"] % patch_size:
            raise ValueError("Invalid backbone or image size")
    paths = ensure_coco_inputs(config, input_paths(config))
    preflight_backbones(config)
    scan(config, paths=paths)


In [ ]:
def capture_notebook():
    inputs = list(get_ipython().history_manager.input_hist_raw[1:])
    sources, definitions, code_nodes = [], {}, {}
    for source in inputs:
        try:
            parsed = ast.parse(source)
        except SyntaxError:
            continue
        # Exclude executed stage calls if this cell is repeated after an interruption.
        is_stage = any(
            isinstance(node, ast.Expr)
            and isinstance(node.value, ast.Call)
            and isinstance(node.value.func, ast.Name)
            and node.value.func.id in {"run_in_workers", "run_training"}
            for node in parsed.body
        )
        if is_stage:
            continue
        sources.append(source)
        for node in parsed.body:
            if isinstance(node, (ast.FunctionDef, ast.ClassDef)):
                definitions[node.name] = ast.unparse(node)
                code_nodes[node.name] = ast.dump(node, include_attributes=False)
            elif isinstance(node, (ast.Assign, ast.AnnAssign)):
                targets = node.targets if isinstance(node, ast.Assign) else [node.target]
                for target in targets:
                    if (
                        isinstance(target, ast.Name)
                        and target.id.isupper()
                        and target.id not in {"SESSION_STARTED", "WORK", "CONFIG_PATH", "CONFIG_YAML", "ACTIVE_CANDIDATES", "TRAIN_END", "SESSION_END"}
                        and not target.id.startswith("NOTEBOOK_")
                        and target.id not in {"RUNTIME_SOURCE", "POLICY_SOURCE"}
                    ):
                        code_nodes[target.id] = ast.dump(node, include_attributes=False)
    runtime_names = [
        "OnnxPrediction",
        "InvalidImageQuality",
        "stable_softmax",
        "scores_v3",
        "decide_v3",
        "PlantDiseaseOnnxClassifier",
    ]
    missing = set(runtime_names) - definitions.keys()
    if missing:
        raise RuntimeError("Required definitions are missing")
    header = (
        "from __future__ import annotations\n"
        "import json\nimport os\nfrom pathlib import Path\n"
        "from dataclasses import dataclass\nfrom typing import Sequence\n"
        "import numpy as np\nfrom PIL import Image, ImageOps\n\n"
    )
    runtime_source = (
        header + "\n\n".join(definitions[name] for name in runtime_names) + "\n"
    )
    policy_source = (
        "import numpy as np\n\n"
        + "\n\n".join(definitions[name] for name in ["scores_v3", "decide_v3"])
        + "\n"
    )
    compile(runtime_source, "onnx_classifier.py", "exec")
    compile(policy_source, "field_v3_policy.py", "exec")
    snapshot = nbformat.v4.new_notebook(
        cells=[nbformat.v4.new_code_cell(source) for source in sources],
        metadata={
            "kernelspec": {
                "display_name": "Python 3",
                "language": "python",
                "name": "python3",
            }
        },
    )
    canonical = json.dumps(code_nodes, sort_keys=True)
    digest = hashlib.sha256(canonical.encode()).hexdigest()
    return digest, nbformat.writes(snapshot), runtime_source, policy_source


In [ ]:
NOTEBOOK_DIGEST, NOTEBOOK_SNAPSHOT, RUNTIME_SOURCE, POLICY_SOURCE = capture_notebook()


## 9. Подготовка выборки

In [ ]:
ACTIVE_CANDIDATES = active_candidates(config)
_, TRAIN_END, _ = deadlines(config, SESSION_STARTED, time.time())
run_in_workers(
    prepare,
    (config,),
    work=WORK,
    label="prepare",
    timeout=max(1.0, TRAIN_END - time.time()),
)
manifest = pd.read_csv(WORK / "split_manifest.csv")
display(pd.crosstab(manifest["split"], manifest["label"]))
assert manifest.groupby("group")["split"].nunique().max() == 1
negative_counts = (
    manifest[manifest.label == "nonplant"].groupby("split").size().to_dict()
)
print("nonplant:", negative_counts)
if not negative_counts:
    print('No nonplant samples')

## 10. Обучение модели

In [ ]:
run_training(config, SESSION_STARTED)
completed = [
    json.loads((WORK / c["name"] / "done.json").read_text())
    for c in ACTIVE_CANDIDATES
    if (WORK / c["name"] / "done.json").exists()
]
display(
    pd.DataFrame(
        [
            {
                "candidate": item["name"],
                "imgsz": item["imgsz"],
                "val_selection_score": item["selection_score"],
            }
            for item in completed
        ]
    )
)


## 11. Сохранение модели

In [ ]:
NOTEBOOK_SNAPSHOT = nbformat.writes(
    nbformat.v4.new_notebook(
        cells=[
            nbformat.v4.new_code_cell(source)
            for source in get_ipython().history_manager.input_hist_raw[1:]
            if source.strip()
        ],
        metadata={
            "kernelspec": {
                "display_name": "Python 3",
                "language": "python",
                "name": "python3",
            }
        },
    )
)
SESSION_END, _, _ = deadlines(config, SESSION_STARTED, time.time())
run_in_workers(
    finalize,
    (config,),
    work=WORK,
    label="finalize",
    timeout=max(1.0, SESSION_END - time.time()),
)


## 12. Результаты и файлы

In [ ]:
artifact = WORK / "release_candidate"
metadata = json.loads((artifact / "model_metadata.json").read_text())
metrics = metadata["evaluation_metrics"]

summary = pd.DataFrame({
    split: {key: value for key, value in values.items()
            if isinstance(value, (int, float)) or value is None}
    for split, values in metrics.items()
})
display(summary)


In [ ]:
class_scores = pd.DataFrame({
    split: values.get("per_class_f1", {}) for split, values in metrics.items()
})
display(class_scores)


In [ ]:
display(FileLink(str(WORK / "hotbed_field_v3.zip")))